# Random Forest · полный ночной эксперимент на GPU
Подключите train_solo_track.parquet. CSV прошлого эксперимента встроен. Запуск на сервере требует cuML, CuPy, pandas, pyarrow, scikit-learn, scipy, optuna, joblib и IPython. Первая ячейка задаёт пути и бюджет.

Подготовка train проверяется на h2 и h6. Optuna подбирает гиперпараметры всех восьми горизонтов. Признаки фиксированы по исходному CSV. Фиксируем лучший набор признаков исходного прогона отдельно для каждого горизонта. На h2 и h6 сравниваем полный и отсечённый train, общую и маршрутную медиану. Каждый вариант проверяется на шести недельных окнах. Затем фиксируем обработку train и запускаем до 10 попыток Optuna для каждого из восьми горизонтов, первая строго повторяет параметры artifacts. Калибровки выбираются на последовательных окнах до test. Независимый test — последние 14 дней. Перезапуск пропускает завершённые обучения. Папка результатов должна соответствовать одному неизменному набору данных и настроек.
Бюджет ориентирован на 8 часов. Новые попытки Optuna прекращаются после 6,5 часов от старта первой ячейки. Начатая попытка заканчивает шесть окон. Оставлен ориентировочный резерв 1,5 часа на остальные горизонты, калибровки, test и сабмит. Время зависит от GPU и диска. По завершении всех этапов сохраняется submission.csv.


In [ ]:
from pathlib import Path
import gc
import json
import re
import tempfile
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.impute import SimpleImputer
import cupy as cp
from cuml.ensemble import RandomForestRegressor

TRAIN_PATH = None
COMPARISON_PATH = None
HORIZONS = (2, 6)
EXPERIMENT_NAMES = ('baseline', 'thin_offsets', 'compact_with_additions')
BASE_PARAMS_BY_HORIZON = {1: {'n_estimators': 150, 'max_depth': 10, 'min_samples_leaf': 32, 'max_features': 0.6, 'max_samples': 0.8}, 2: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 24, 'max_features': 0.3406585285177396, 'max_samples': 0.9330880728874675}, 3: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 24, 'max_features': 0.3406585285177396, 'max_samples': 0.9330880728874675}, 4: {'n_estimators': 150, 'max_depth': 10, 'min_samples_leaf': 32, 'max_features': 0.6, 'max_samples': 0.8}, 5: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}, 6: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 23, 'max_features': 0.5045012539746527, 'max_samples': 0.6831809216468459}, 7: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}, 8: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}}
SEED = 42
N_STREAMS = 4
PREDICTION_BATCH = 32768
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TRAIN_MINUTES = (570, 600, 630, 660, 690)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
RESIDUAL = False

import hashlib
import joblib
import optuna
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression

TRAIN_PATH = Path('/kaggle/input/datasets/moonshyxd/case-rwb/train_solo_track.parquet')
RESULTS = Path('/kaggle/working/rf_night_8h_fixed_features') if Path('/kaggle/working').exists() else Path('rf_night_8h_fixed_features')
N_TRIALS = 10
RUN_STARTED = time.monotonic()
SEARCH_DEADLINE = RUN_STARTED + 6.5 * 3600
RESULTS.mkdir(parents=True, exist_ok=True)
if not TRAIN_PATH.is_file():
    matches = list(Path('/kaggle/input').rglob('train_solo_track.parquet'))
    if len(matches) != 1:
        raise FileNotFoundError('Укажите TRAIN_PATH в первой ячейке.')
    TRAIN_PATH = matches[0]
cp.zeros(1, dtype=cp.float32)
cp.cuda.get_current_stream().synchronize()
OUT = RESULTS / 'features'
OUT.mkdir(exist_ok=True)

def save_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.partial')
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2))
    temporary.replace(path)

def save_frame(path, frame):
    temporary = path.with_suffix('.partial')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

digest = hashlib.sha256()
with TRAIN_PATH.open('rb') as stream:
    for block in iter(lambda: stream.read(8*1024*1024), b''):
        digest.update(block)
contract = dict(version=1, data=digest.hexdigest(), params=BASE_PARAMS_BY_HORIZON, seed=SEED, streams=N_STREAMS)
contract = json.loads(json.dumps(contract))
path = RESULTS / 'contract.json'
if path.exists() and json.loads(path.read_text()) != contract:
    raise ValueError('Используйте новую RESULTS для изменённых данных или настроек.')
save_json(path, contract)


In [ ]:
previous = pd.DataFrame(json.loads('[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.256588,"Relative Bias":-0.000261,"|Relative Bias|":0.000261,"метрика конкурса":0.256849,"среднее по дням":0.264723,"variant":"direct","experiment":"baseline","horizon":1,"fold":1,"features":388,"volume":29748780000.0,"absolute_error":7633171000.0,"signed_error":-7766953.0,"seconds":26.22239},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.262693,"Relative Bias":0.005146,"|Relative Bias|":0.005146,"метрика конкурса":0.267838,"среднее по дням":0.288929,"variant":"direct","experiment":"baseline","horizon":1,"fold":1,"features":388,"volume":1932004000.0,"absolute_error":507523200.0,"signed_error":9941647.0,"seconds":26.245869},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.258224,"Relative Bias":-0.000313,"|Relative Bias|":0.000313,"метрика конкурса":0.258536,"среднее по дням":0.266892,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":1,"features":232,"volume":29748780000.0,"absolute_error":7681844000.0,"signed_error":-9298659.0,"seconds":9.917906},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.262388,"Relative Bias":0.004355,"|Relative Bias|":0.004355,"метрика конкурса":0.266743,"среднее по дням":0.288279,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":1,"features":232,"volume":1932004000.0,"absolute_error":506934700.0,"signed_error":8413071.0,"seconds":9.933219},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.257013,"Relative Bias":-0.000266,"|Relative Bias|":0.000266,"метрика конкурса":0.257279,"среднее по дням":0.264974,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":1,"features":245,"volume":29748780000.0,"absolute_error":7645828000.0,"signed_error":-7922351.0,"seconds":10.176453},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.262095,"Relative Bias":0.006041,"|Relative Bias|":0.006041,"метрика конкурса":0.268136,"среднее по дням":0.289183,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":1,"features":245,"volume":1932004000.0,"absolute_error":506367700.0,"signed_error":11672040.0,"seconds":10.187476},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.257883,"Relative Bias":-0.000292,"|Relative Bias|":0.000292,"метрика конкурса":0.258175,"среднее по дням":0.266516,"variant":"direct","experiment":"baseline","horizon":1,"fold":2,"features":388,"volume":38649630000.0,"absolute_error":9967092000.0,"signed_error":-11277700.0,"seconds":33.672311},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.283222,"Relative Bias":0.066929,"|Relative Bias|":0.066929,"метрика конкурса":0.350151,"среднее по дням":0.365136,"variant":"direct","experiment":"baseline","horizon":1,"fold":2,"features":388,"volume":1903508000.0,"absolute_error":539115000.0,"signed_error":127399400.0,"seconds":33.686869},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.259491,"Relative Bias":-0.000252,"|Relative Bias|":0.000252,"метрика конкурса":0.259743,"среднее по дням":0.269175,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":2,"features":232,"volume":38649630000.0,"absolute_error":10029240000.0,"signed_error":-9750086.0,"seconds":14.976312},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.282792,"Relative Bias":0.064997,"|Relative Bias|":0.064997,"метрика конкурса":0.34779,"среднее по дням":0.362752,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":2,"features":232,"volume":1903508000.0,"absolute_error":538297500.0,"signed_error":123723200.0,"seconds":14.986894},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.25828,"Relative Bias":-0.000236,"|Relative Bias|":0.000236,"метрика конкурса":0.258516,"среднее по дням":0.266571,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":2,"features":245,"volume":38649630000.0,"absolute_error":9982411000.0,"signed_error":-9129765.0,"seconds":15.221267},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.282914,"Relative Bias":0.067732,"|Relative Bias|":0.067732,"метрика конкурса":0.350646,"среднее по дням":0.36579,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":2,"features":245,"volume":1903508000.0,"absolute_error":538530100.0,"signed_error":128927800.0,"seconds":15.232375},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.258536,"Relative Bias":-0.000325,"|Relative Bias|":0.000325,"метрика конкурса":0.258861,"среднее по дням":0.268499,"variant":"direct","experiment":"baseline","horizon":1,"fold":3,"features":388,"volume":48027310000.0,"absolute_error":12416790000.0,"signed_error":-15625660.0,"seconds":41.941955},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.264405,"Relative Bias":-0.01111,"|Relative Bias|":0.01111,"метрика конкурса":0.275516,"среднее по дням":0.281283,"variant":"direct","experiment":"baseline","horizon":1,"fold":3,"features":388,"volume":2053871000.0,"absolute_error":543054000.0,"signed_error":-22819470.0,"seconds":41.956384},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.259918,"Relative Bias":-0.000364,"|Relative Bias|":0.000364,"метрика конкурса":0.260282,"среднее по дням":0.270764,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":3,"features":232,"volume":48027310000.0,"absolute_error":12483160000.0,"signed_error":-17488320.0,"seconds":18.321837},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.264843,"Relative Bias":-0.009796,"|Relative Bias|":0.009796,"метрика конкурса":0.27464,"среднее по дням":0.281483,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":3,"features":232,"volume":2053871000.0,"absolute_error":543954000.0,"signed_error":-20120260.0,"seconds":18.333916},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.258686,"Relative Bias":-0.000309,"|Relative Bias|":0.000309,"метрика конкурса":0.258994,"среднее по дням":0.268537,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":3,"features":245,"volume":48027310000.0,"absolute_error":12423970000.0,"signed_error":-14826220.0,"seconds":17.456716},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.263485,"Relative Bias":-0.007273,"|Relative Bias|":0.007273,"метрика конкурса":0.270758,"среднее по дням":0.278013,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":3,"features":245,"volume":2053871000.0,"absolute_error":541164400.0,"signed_error":-14938430.0,"seconds":17.468464},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.259515,"Relative Bias":-0.000144,"|Relative Bias|":0.000144,"метрика конкурса":0.259659,"среднее по дням":0.270522,"variant":"direct","experiment":"baseline","horizon":1,"fold":4,"features":388,"volume":57620340000.0,"absolute_error":14953370000.0,"signed_error":-8274031.0,"seconds":48.856591},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.259797,"Relative Bias":-0.036723,"|Relative Bias|":0.036723,"метрика конкурса":0.29652,"среднее по дням":0.295791,"variant":"direct","experiment":"baseline","horizon":1,"fold":4,"features":388,"volume":2063192000.0,"absolute_error":536011200.0,"signed_error":-75767570.0,"seconds":48.871821},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.260822,"Relative Bias":-0.00017,"|Relative Bias|":0.00017,"метрика конкурса":0.260993,"среднее по дням":0.272434,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":4,"features":232,"volume":57620340000.0,"absolute_error":15028670000.0,"signed_error":-9806383.0,"seconds":21.881792},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.260363,"Relative Bias":-0.029393,"|Relative Bias|":0.029393,"метрика конкурса":0.289756,"среднее по дням":0.291913,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":4,"features":232,"volume":2063192000.0,"absolute_error":537179000.0,"signed_error":-60642810.0,"seconds":21.892805},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.259667,"Relative Bias":-0.000136,"|Relative Bias|":0.000136,"метрика конкурса":0.259803,"среднее по дням":0.271286,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":4,"features":245,"volume":57620340000.0,"absolute_error":14962080000.0,"signed_error":-7846767.0,"seconds":20.324936},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.258985,"Relative Bias":-0.032566,"|Relative Bias|":0.032566,"метрика конкурса":0.291551,"среднее по дням":0.29216,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":4,"features":245,"volume":2063192000.0,"absolute_error":534335700.0,"signed_error":-67189490.0,"seconds":20.336294},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.260327,"Relative Bias":-0.000313,"|Relative Bias|":0.000313,"метрика конкурса":0.260639,"среднее по дням":0.272319,"variant":"direct","experiment":"baseline","horizon":1,"fold":5,"features":388,"volume":67659790000.0,"absolute_error":17613670000.0,"signed_error":-21147150.0,"seconds":58.254558},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.268209,"Relative Bias":-0.006437,"|Relative Bias|":0.006437,"метрика конкурса":0.274646,"среднее по дням":0.291013,"variant":"direct","experiment":"baseline","horizon":1,"fold":5,"features":388,"volume":2075688000.0,"absolute_error":556718600.0,"signed_error":-13361290.0,"seconds":58.268711},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.26154,"Relative Bias":-0.000271,"|Relative Bias|":0.000271,"метрика конкурса":0.261811,"среднее по дням":0.274039,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":5,"features":232,"volume":67659790000.0,"absolute_error":17695760000.0,"signed_error":-18347970.0,"seconds":23.771553},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.26948,"Relative Bias":-0.002641,"|Relative Bias|":0.002641,"метрика конкурса":0.272122,"среднее по дням":0.293276,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":5,"features":232,"volume":2075688000.0,"absolute_error":559356900.0,"signed_error":-5482548.0,"seconds":23.782195},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.26029,"Relative Bias":-0.000195,"|Relative Bias|":0.000195,"метрика конкурса":0.260485,"среднее по дням":0.272435,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":5,"features":245,"volume":67659790000.0,"absolute_error":17611180000.0,"signed_error":-13193000.0,"seconds":23.176182},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.267897,"Relative Bias":-0.003575,"|Relative Bias|":0.003575,"метрика конкурса":0.271471,"среднее по дням":0.291541,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":5,"features":245,"volume":2075688000.0,"absolute_error":556070000.0,"signed_error":-7419993.0,"seconds":23.187292},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.260789,"Relative Bias":-0.000214,"|Relative Bias|":0.000214,"метрика конкурса":0.261002,"среднее по дням":0.274907,"variant":"direct","experiment":"baseline","horizon":1,"fold":6,"features":388,"volume":77733690000.0,"absolute_error":20272060000.0,"signed_error":-16621450.0,"seconds":66.921942},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.27028,"Relative Bias":-0.00915,"|Relative Bias|":0.00915,"метрика конкурса":0.27943,"среднее по дням":0.290103,"variant":"direct","experiment":"baseline","horizon":1,"fold":6,"features":388,"volume":2084706000.0,"absolute_error":563454000.0,"signed_error":-19075960.0,"seconds":66.93644},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.261845,"Relative Bias":-0.000248,"|Relative Bias|":0.000248,"метрика конкурса":0.262094,"среднее по дням":0.275854,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":6,"features":232,"volume":77733690000.0,"absolute_error":20354210000.0,"signed_error":-19295440.0,"seconds":26.577312},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.271777,"Relative Bias":-0.003451,"|Relative Bias|":0.003451,"метрика конкурса":0.275228,"среднее по дням":0.293399,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":6,"features":232,"volume":2084706000.0,"absolute_error":566574000.0,"signed_error":-7195277.0,"seconds":26.588443},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.260668,"Relative Bias":-0.000156,"|Relative Bias|":0.000156,"метрика конкурса":0.260823,"среднее по дням":0.274446,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":6,"features":245,"volume":77733690000.0,"absolute_error":20262680000.0,"signed_error":-12087990.0,"seconds":27.515281},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.269882,"Relative Bias":-0.008092,"|Relative Bias|":0.008092,"метрика конкурса":0.277974,"среднее по дням":0.290694,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":6,"features":245,"volume":2084706000.0,"absolute_error":562624900.0,"signed_error":-16868680.0,"seconds":27.525926},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.322831,"Relative Bias":-0.000087,"|Relative Bias|":0.000087,"метрика конкурса":0.322918,"среднее по дням":0.337179,"variant":"direct","experiment":"baseline","horizon":2,"fold":1,"features":390,"volume":30245590000.0,"absolute_error":9764216000.0,"signed_error":-2631093.0,"seconds":25.698697},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325589,"Relative Bias":0.006297,"|Relative Bias|":0.006297,"метрика конкурса":0.331886,"среднее по дням":0.344449,"variant":"direct","experiment":"baseline","horizon":2,"fold":1,"features":390,"volume":1915804000.0,"absolute_error":623765300.0,"signed_error":12064080.0,"seconds":25.713429},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.323415,"Relative Bias":-0.000126,"|Relative Bias|":0.000126,"метрика конкурса":0.32354,"среднее по дням":0.338257,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":1,"features":234,"volume":30245590000.0,"absolute_error":9781864000.0,"signed_error":-3801653.0,"seconds":11.521686},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325618,"Relative Bias":0.005353,"|Relative Bias|":0.005353,"метрика конкурса":0.330971,"среднее по дням":0.345024,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":1,"features":234,"volume":1915804000.0,"absolute_error":623820800.0,"signed_error":10255820.0,"seconds":11.532506},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.321371,"Relative Bias":-0.000128,"|Relative Bias|":0.000128,"метрика конкурса":0.321499,"среднее по дням":0.33589,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":1,"features":247,"volume":30245590000.0,"absolute_error":9720049000.0,"signed_error":-3882704.0,"seconds":10.958094},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324349,"Relative Bias":0.009249,"|Relative Bias|":0.009249,"метрика конкурса":0.333599,"среднее по дням":0.346471,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":1,"features":247,"volume":1915804000.0,"absolute_error":621389700.0,"signed_error":17719700.0,"seconds":10.969433},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.322139,"Relative Bias":-0.000287,"|Relative Bias|":0.000287,"метрика конкурса":0.322426,"среднее по дням":0.33868,"variant":"direct","experiment":"baseline","horizon":2,"fold":2,"features":390,"volume":39292040000.0,"absolute_error":12657480000.0,"signed_error":-11290050.0,"seconds":31.786005},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330877,"Relative Bias":0.056523,"|Relative Bias|":0.056523,"метрика конкурса":0.3874,"среднее по дням":0.395405,"variant":"direct","experiment":"baseline","horizon":2,"fold":2,"features":390,"volume":1933559000.0,"absolute_error":639770300.0,"signed_error":109291100.0,"seconds":31.800458},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.322582,"Relative Bias":-0.000275,"|Relative Bias|":0.000275,"метрика конкурса":0.322857,"среднее по дням":0.339777,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":2,"features":234,"volume":39292040000.0,"absolute_error":12674890000.0,"signed_error":-10819120.0,"seconds":14.40306},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330706,"Relative Bias":0.056268,"|Relative Bias|":0.056268,"метрика конкурса":0.386974,"среднее по дням":0.394984,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":2,"features":234,"volume":1933559000.0,"absolute_error":639439800.0,"signed_error":108798100.0,"seconds":14.414307},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.320532,"Relative Bias":-0.00027,"|Relative Bias|":0.00027,"метрика конкурса":0.320803,"среднее по дням":0.336607,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":2,"features":247,"volume":39292040000.0,"absolute_error":12594370000.0,"signed_error":-10623560.0,"seconds":14.296427},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330885,"Relative Bias":0.066045,"|Relative Bias|":0.066045,"метрика конкурса":0.39693,"среднее по дням":0.403121,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":2,"features":247,"volume":1933559000.0,"absolute_error":639785800.0,"signed_error":127701700.0,"seconds":14.307703},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.320997,"Relative Bias":-0.000197,"|Relative Bias|":0.000197,"метрика конкурса":0.321195,"среднее по дням":0.338819,"variant":"direct","experiment":"baseline","horizon":2,"fold":3,"features":390,"volume":48881010000.0,"absolute_error":15690670000.0,"signed_error":-9640998.0,"seconds":38.937374},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.314283,"Relative Bias":-0.039334,"|Relative Bias|":0.039334,"метрика конкурса":0.353616,"среднее по дням":0.364575,"variant":"direct","experiment":"baseline","horizon":2,"fold":3,"features":390,"volume":2101718000.0,"absolute_error":660533400.0,"signed_error":-82668000.0,"seconds":38.952366},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.321237,"Relative Bias":-0.000186,"|Relative Bias|":0.000186,"метрика конкурса":0.321424,"среднее по дням":0.33959,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":3,"features":234,"volume":48881010000.0,"absolute_error":15702400000.0,"signed_error":-9109512.0,"seconds":17.409773},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.314248,"Relative Bias":-0.038177,"|Relative Bias|":0.038177,"метрика конкурса":0.352425,"среднее по дням":0.363744,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":3,"features":234,"volume":2101718000.0,"absolute_error":660460800.0,"signed_error":-80237770.0,"seconds":17.421314},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.319291,"Relative Bias":-0.00022,"|Relative Bias|":0.00022,"метрика конкурса":0.319511,"среднее по дням":0.337395,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":3,"features":247,"volume":48881010000.0,"absolute_error":15607250000.0,"signed_error":-10765940.0,"seconds":17.077481},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.312016,"Relative Bias":-0.030491,"|Relative Bias|":0.030491,"метрика конкурса":0.342507,"среднее по дням":0.355148,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":3,"features":247,"volume":2101718000.0,"absolute_error":655769200.0,"signed_error":-64084220.0,"seconds":17.088973},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.32077,"Relative Bias":-0.000112,"|Relative Bias|":0.000112,"метрика конкурса":0.320882,"среднее по дням":0.339687,"variant":"direct","experiment":"baseline","horizon":2,"fold":4,"features":390,"volume":58796850000.0,"absolute_error":18860250000.0,"signed_error":-6578303.0,"seconds":46.300561},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325713,"Relative Bias":-0.03971,"|Relative Bias|":0.03971,"метрика конкурса":0.365423,"среднее по дням":0.374232,"variant":"direct","experiment":"baseline","horizon":2,"fold":4,"features":390,"volume":2057112000.0,"absolute_error":670028800.0,"signed_error":-81688010.0,"seconds":46.315267},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.320991,"Relative Bias":-0.000082,"|Relative Bias|":0.000082,"метрика конкурса":0.321072,"среднее по дням":0.34051,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":4,"features":234,"volume":58796850000.0,"absolute_error":18873240000.0,"signed_error":-4801584.0,"seconds":20.732989},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325618,"Relative Bias":-0.036691,"|Relative Bias|":0.036691,"метрика конкурса":0.362308,"среднее по дням":0.373428,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":4,"features":234,"volume":2057112000.0,"absolute_error":669831800.0,"signed_error":-75476730.0,"seconds":20.744054},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.319212,"Relative Bias":-0.00011,"|Relative Bias|":0.00011,"метрика конкурса":0.319322,"среднее по дням":0.338986,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":4,"features":247,"volume":58796850000.0,"absolute_error":18768650000.0,"signed_error":-6490754.0,"seconds":20.025272},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323687,"Relative Bias":-0.037542,"|Relative Bias|":0.037542,"метрика конкурса":0.361229,"среднее по дням":0.371178,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":4,"features":247,"volume":2057112000.0,"absolute_error":665860800.0,"signed_error":-77227730.0,"seconds":20.036736},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.320721,"Relative Bias":-0.000116,"|Relative Bias|":0.000116,"метрика конкурса":0.320837,"среднее по дням":0.341301,"variant":"direct","experiment":"baseline","horizon":2,"fold":5,"features":390,"volume":69126560000.0,"absolute_error":22170370000.0,"signed_error":-8020000.0,"seconds":53.884605},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.317186,"Relative Bias":-0.015417,"|Relative Bias|":0.015417,"метрика конкурса":0.332603,"среднее по дням":0.342247,"variant":"direct","experiment":"baseline","horizon":2,"fold":5,"features":390,"volume":2118564000.0,"absolute_error":671978800.0,"signed_error":-32661950.0,"seconds":53.899636},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.320847,"Relative Bias":-0.000114,"|Relative Bias|":0.000114,"метрика конкурса":0.320962,"среднее по дням":0.341898,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":5,"features":234,"volume":69126560000.0,"absolute_error":22179070000.0,"signed_error":-7901577.0,"seconds":23.530924},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.317211,"Relative Bias":-0.013345,"|Relative Bias|":0.013345,"метрика конкурса":0.330556,"среднее по дням":0.341094,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":5,"features":234,"volume":2118564000.0,"absolute_error":672031800.0,"signed_error":-28272670.0,"seconds":23.542064},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.319037,"Relative Bias":-0.000124,"|Relative Bias|":0.000124,"метрика конкурса":0.31916,"среднее по дням":0.340092,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":5,"features":247,"volume":69126560000.0,"absolute_error":22053900000.0,"signed_error":-8554299.0,"seconds":23.178056},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.315049,"Relative Bias":-0.013012,"|Relative Bias|":0.013012,"метрика конкурса":0.328061,"среднее по дням":0.339498,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":5,"features":247,"volume":2118564000.0,"absolute_error":667452100.0,"signed_error":-27566160.0,"seconds":23.18974},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.321427,"Relative Bias":-0.000107,"|Relative Bias|":0.000107,"метрика конкурса":0.321534,"среднее по дням":0.344727,"variant":"direct","experiment":"baseline","horizon":2,"fold":6,"features":390,"volume":79480170000.0,"absolute_error":25547080000.0,"signed_error":-8492330.0,"seconds":62.926732},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322175,"Relative Bias":-0.00411,"|Relative Bias|":0.00411,"метрика конкурса":0.326285,"среднее по дням":0.343481,"variant":"direct","experiment":"baseline","horizon":2,"fold":6,"features":390,"volume":2122104000.0,"absolute_error":683689800.0,"signed_error":-8721293.0,"seconds":62.941918},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.321518,"Relative Bias":-0.000099,"|Relative Bias|":0.000099,"метрика конкурса":0.321617,"среднее по дням":0.344884,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":6,"features":234,"volume":79480170000.0,"absolute_error":25554310000.0,"signed_error":-7876271.0,"seconds":26.348142},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322423,"Relative Bias":-0.001597,"|Relative Bias|":0.001597,"метрика конкурса":0.32402,"среднее по дням":0.343666,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":6,"features":234,"volume":2122104000.0,"absolute_error":684216000.0,"signed_error":-3388501.0,"seconds":26.358649},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.319735,"Relative Bias":-0.000088,"|Relative Bias|":0.000088,"метрика конкурса":0.319823,"среднее по дням":0.343264,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":6,"features":247,"volume":79480170000.0,"absolute_error":25412590000.0,"signed_error":-6973333.0,"seconds":26.802039},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.320224,"Relative Bias":-0.005358,"|Relative Bias|":0.005358,"метрика конкурса":0.325582,"среднее по дням":0.343564,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":6,"features":247,"volume":2122104000.0,"absolute_error":679549700.0,"signed_error":-11370250.0,"seconds":26.813156},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.334374,"Relative Bias":-0.000204,"|Relative Bias|":0.000204,"метрика конкурса":0.334578,"среднее по дням":0.34946,"variant":"direct","experiment":"baseline","horizon":3,"fold":1,"features":392,"volume":30060130000.0,"absolute_error":10051320000.0,"signed_error":-6143330.0,"seconds":25.961082},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.321662,"Relative Bias":0.00286,"|Relative Bias|":0.00286,"метрика конкурса":0.324522,"среднее по дням":0.343846,"variant":"direct","experiment":"baseline","horizon":3,"fold":1,"features":392,"volume":1931182000.0,"absolute_error":621188200.0,"signed_error":5522907.0,"seconds":25.976591},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.334701,"Relative Bias":-0.000257,"|Relative Bias|":0.000257,"метрика конкурса":0.334958,"среднее по дням":0.349629,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":1,"features":236,"volume":30060130000.0,"absolute_error":10061150000.0,"signed_error":-7718776.0,"seconds":11.569985},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322041,"Relative Bias":0.004142,"|Relative Bias|":0.004142,"метрика конкурса":0.326183,"среднее по дням":0.344883,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":1,"features":236,"volume":1931182000.0,"absolute_error":621919800.0,"signed_error":7999500.0,"seconds":11.580688},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.333435,"Relative Bias":-0.00023,"|Relative Bias|":0.00023,"метрика конкурса":0.333664,"среднее по дням":0.349937,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":1,"features":249,"volume":30060130000.0,"absolute_error":10023090000.0,"signed_error":-6904869.0,"seconds":11.043004},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.321068,"Relative Bias":0.007096,"|Relative Bias|":0.007096,"метрика конкурса":0.328163,"среднее по дням":0.348329,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":1,"features":249,"volume":1931182000.0,"absolute_error":620039900.0,"signed_error":13702780.0,"seconds":11.054471},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.333386,"Relative Bias":-0.000282,"|Relative Bias|":0.000282,"метрика конкурса":0.333669,"среднее по дням":0.349003,"variant":"direct","experiment":"baseline","horizon":3,"fold":2,"features":392,"volume":39081400000.0,"absolute_error":13029210000.0,"signed_error":-11038620.0,"seconds":32.005029},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.316772,"Relative Bias":-0.00223,"|Relative Bias|":0.00223,"метрика конкурса":0.319003,"среднее по дням":0.361776,"variant":"direct","experiment":"baseline","horizon":3,"fold":2,"features":392,"volume":2064414000.0,"absolute_error":653949600.0,"signed_error":-4604600.0,"seconds":32.020142},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.333623,"Relative Bias":-0.000245,"|Relative Bias|":0.000245,"метрика конкурса":0.333868,"среднее по дням":0.349654,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":2,"features":236,"volume":39081400000.0,"absolute_error":13038450000.0,"signed_error":-9590512.0,"seconds":14.512181},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.316888,"Relative Bias":-0.001118,"|Relative Bias|":0.001118,"метрика конкурса":0.318006,"среднее по дням":0.361741,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":2,"features":236,"volume":2064414000.0,"absolute_error":654187800.0,"signed_error":-2307710.0,"seconds":14.52361},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.332387,"Relative Bias":-0.000251,"|Relative Bias|":0.000251,"метрика конкурса":0.332638,"среднее по дням":0.348604,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":2,"features":249,"volume":39081400000.0,"absolute_error":12990150000.0,"signed_error":-9802305.0,"seconds":14.201775},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.315856,"Relative Bias":0.004984,"|Relative Bias|":0.004984,"метрика конкурса":0.32084,"среднее по дням":0.362536,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":2,"features":249,"volume":2064414000.0,"absolute_error":652057100.0,"signed_error":10288800.0,"seconds":14.212978},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.33174,"Relative Bias":-0.000179,"|Relative Bias|":0.000179,"метрика конкурса":0.33192,"среднее по дням":0.347899,"variant":"direct","experiment":"baseline","horizon":3,"fold":3,"features":392,"volume":48698000000.0,"absolute_error":16155090000.0,"signed_error":-8734305.0,"seconds":38.90073},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325419,"Relative Bias":-0.034284,"|Relative Bias|":0.034284,"метрика конкурса":0.359703,"среднее по дням":0.371215,"variant":"direct","experiment":"baseline","horizon":3,"fold":3,"features":392,"volume":2118422000.0,"absolute_error":689374500.0,"signed_error":-72627190.0,"seconds":38.915511},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.332007,"Relative Bias":-0.000173,"|Relative Bias|":0.000173,"метрика конкурса":0.33218,"среднее по дням":0.348451,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":3,"features":236,"volume":48698000000.0,"absolute_error":16168090000.0,"signed_error":-8415185.0,"seconds":17.553912},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325634,"Relative Bias":-0.031363,"|Relative Bias|":0.031363,"метрика конкурса":0.356997,"среднее по дням":0.36939,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":3,"features":236,"volume":2118422000.0,"absolute_error":689829300.0,"signed_error":-66440050.0,"seconds":17.565926},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.330804,"Relative Bias":-0.00018,"|Relative Bias|":0.00018,"метрика конкурса":0.330984,"среднее по дням":0.348013,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":3,"features":249,"volume":48698000000.0,"absolute_error":16109490000.0,"signed_error":-8779546.0,"seconds":16.999582},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324496,"Relative Bias":-0.026203,"|Relative Bias|":0.026203,"метрика конкурса":0.350699,"среднее по дням":0.362524,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":3,"features":249,"volume":2118422000.0,"absolute_error":687419000.0,"signed_error":-55509570.0,"seconds":17.011177},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.331085,"Relative Bias":-0.000079,"|Relative Bias|":0.000079,"метрика конкурса":0.331165,"среднее по дням":0.347925,"variant":"direct","experiment":"baseline","horizon":3,"fold":4,"features":392,"volume":58727020000.0,"absolute_error":19443670000.0,"signed_error":-4647942.0,"seconds":46.243536},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.328504,"Relative Bias":-0.056152,"|Relative Bias|":0.056152,"метрика конкурса":0.384656,"среднее по дням":0.384134,"variant":"direct","experiment":"baseline","horizon":3,"fold":4,"features":392,"volume":2120046000.0,"absolute_error":696444500.0,"signed_error":-119044300.0,"seconds":46.258174},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.331315,"Relative Bias":-0.000081,"|Relative Bias|":0.000081,"метрика конкурса":0.331396,"среднее по дням":0.348371,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":4,"features":236,"volume":58727020000.0,"absolute_error":19457160000.0,"signed_error":-4744659.0,"seconds":20.941715},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.328281,"Relative Bias":-0.051016,"|Relative Bias|":0.051016,"метрика конкурса":0.379297,"среднее по дням":0.378762,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":4,"features":236,"volume":2120046000.0,"absolute_error":695970400.0,"signed_error":-108156300.0,"seconds":20.952528},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.330263,"Relative Bias":-0.000049,"|Relative Bias|":0.000049,"метрика конкурса":0.330311,"среднее по дням":0.348337,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":4,"features":249,"volume":58727020000.0,"absolute_error":19395340000.0,"signed_error":-2863123.0,"seconds":19.975772},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326993,"Relative Bias":-0.050668,"|Relative Bias|":0.050668,"метрика конкурса":0.377661,"среднее по дням":0.377153,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":4,"features":249,"volume":2120046000.0,"absolute_error":693240400.0,"signed_error":-107418200.0,"seconds":19.987443},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.330459,"Relative Bias":-0.000095,"|Relative Bias|":0.000095,"метрика конкурса":0.330554,"среднее по дням":0.349168,"variant":"direct","experiment":"baseline","horizon":3,"fold":5,"features":392,"volume":69199000000.0,"absolute_error":22867440000.0,"signed_error":-6545585.0,"seconds":54.890009},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325333,"Relative Bias":-0.018886,"|Relative Bias|":0.018886,"метрика конкурса":0.344219,"среднее по дням":0.363458,"variant":"direct","experiment":"baseline","horizon":3,"fold":5,"features":392,"volume":2146331000.0,"absolute_error":698271500.0,"signed_error":-40535550.0,"seconds":54.905517},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.330625,"Relative Bias":-0.000121,"|Relative Bias|":0.000121,"метрика конкурса":0.330746,"среднее по дням":0.349659,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":5,"features":236,"volume":69199000000.0,"absolute_error":22878960000.0,"signed_error":-8347499.0,"seconds":23.670605},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325218,"Relative Bias":-0.014899,"|Relative Bias|":0.014899,"метрика конкурса":0.340118,"среднее по дням":0.361716,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":5,"features":236,"volume":2146331000.0,"absolute_error":698026000.0,"signed_error":-31979150.0,"seconds":23.681039},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.329533,"Relative Bias":-0.000082,"|Relative Bias|":0.000082,"метрика конкурса":0.329616,"среднее по дням":0.349218,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":5,"features":249,"volume":69199000000.0,"absolute_error":22803370000.0,"signed_error":-5705204.0,"seconds":23.189139},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32291,"Relative Bias":-0.013385,"|Relative Bias|":0.013385,"метрика конкурса":0.336295,"среднее по дням":0.357733,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":5,"features":249,"volume":2146331000.0,"absolute_error":693072200.0,"signed_error":-28728760.0,"seconds":23.20123},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.330477,"Relative Bias":-0.000071,"|Relative Bias|":0.000071,"метрика конкурса":0.330547,"среднее по дням":0.352059,"variant":"direct","experiment":"baseline","horizon":3,"fold":6,"features":392,"volume":79671210000.0,"absolute_error":26329490000.0,"signed_error":-5627102.0,"seconds":64.118343},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.332637,"Relative Bias":-0.002148,"|Relative Bias|":0.002148,"метрика конкурса":0.334785,"среднее по дням":0.352283,"variant":"direct","experiment":"baseline","horizon":3,"fold":6,"features":392,"volume":2115532000.0,"absolute_error":703703400.0,"signed_error":-4544340.0,"seconds":64.13321},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.330677,"Relative Bias":-0.000078,"|Relative Bias|":0.000078,"метрика конкурса":0.330754,"среднее по дням":0.351964,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":6,"features":236,"volume":79671210000.0,"absolute_error":26345400000.0,"signed_error":-6191355.0,"seconds":26.420847},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.333133,"Relative Bias":0.002894,"|Relative Bias|":0.002894,"метрика конкурса":0.336027,"среднее по дням":0.351852,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":6,"features":236,"volume":2115532000.0,"absolute_error":704753000.0,"signed_error":6121876.0,"seconds":26.431438},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.329633,"Relative Bias":-0.000068,"|Relative Bias|":0.000068,"метрика конкурса":0.3297,"среднее по дням":0.351936,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":6,"features":249,"volume":79671210000.0,"absolute_error":26262230000.0,"signed_error":-5396057.0,"seconds":26.960539},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.331696,"Relative Bias":-0.002196,"|Relative Bias|":0.002196,"метрика конкурса":0.333893,"среднее по дням":0.352483,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":6,"features":249,"volume":2115532000.0,"absolute_error":701714000.0,"signed_error":-4646208.0,"seconds":26.972029},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.333579,"Relative Bias":-0.000558,"|Relative Bias|":0.000558,"метрика конкурса":0.334137,"среднее по дням":0.350729,"variant":"direct","experiment":"baseline","horizon":4,"fold":1,"features":394,"volume":29488060000.0,"absolute_error":9836599000.0,"signed_error":-16463920.0,"seconds":27.221965},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324309,"Relative Bias":-0.000842,"|Relative Bias|":0.000842,"метрика конкурса":0.325151,"среднее по дням":0.351739,"variant":"direct","experiment":"baseline","horizon":4,"fold":1,"features":394,"volume":1942286000.0,"absolute_error":629900000.0,"signed_error":-1636082.0,"seconds":27.238045},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.334435,"Relative Bias":-0.000614,"|Relative Bias|":0.000614,"метрика конкурса":0.335048,"среднее по дням":0.351722,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":1,"features":238,"volume":29488060000.0,"absolute_error":9861834000.0,"signed_error":-18090940.0,"seconds":12.014565},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324067,"Relative Bias":-0.002009,"|Relative Bias|":0.002009,"метрика конкурса":0.326076,"среднее по дням":0.351771,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":1,"features":238,"volume":1942286000.0,"absolute_error":629430200.0,"signed_error":-3902229.0,"seconds":12.025546},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.333233,"Relative Bias":-0.000523,"|Relative Bias|":0.000523,"метрика конкурса":0.333756,"среднее по дням":0.353167,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":1,"features":251,"volume":29488060000.0,"absolute_error":9826409000.0,"signed_error":-15418770.0,"seconds":11.548206},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323594,"Relative Bias":-0.000046,"|Relative Bias|":0.000046,"метрика конкурса":0.323641,"среднее по дням":0.353373,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":1,"features":251,"volume":1942286000.0,"absolute_error":628513100.0,"signed_error":-89680.4,"seconds":11.560219},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.332477,"Relative Bias":-0.000183,"|Relative Bias|":0.000183,"метрика конкурса":0.33266,"среднее по дням":0.349236,"variant":"direct","experiment":"baseline","horizon":4,"fold":2,"features":394,"volume":38430030000.0,"absolute_error":12777090000.0,"signed_error":-7039764.0,"seconds":32.941259},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323681,"Relative Bias":-0.003729,"|Relative Bias|":0.003729,"метрика конкурса":0.32741,"среднее по дням":0.367881,"variant":"direct","experiment":"baseline","horizon":4,"fold":2,"features":394,"volume":2089553000.0,"absolute_error":676348100.0,"signed_error":-7792131.0,"seconds":32.955514},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.333198,"Relative Bias":-0.000189,"|Relative Bias|":0.000189,"метрика конкурса":0.333386,"среднее по дням":0.349968,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":2,"features":238,"volume":38430030000.0,"absolute_error":12804800000.0,"signed_error":-7244980.0,"seconds":15.544521},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323286,"Relative Bias":-0.005009,"|Relative Bias|":0.005009,"метрика конкурса":0.328295,"среднее по дням":0.367181,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":2,"features":238,"volume":2089553000.0,"absolute_error":675523700.0,"signed_error":-10466880.0,"seconds":15.555738},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.332045,"Relative Bias":-0.000092,"|Relative Bias|":0.000092,"метрика конкурса":0.332137,"среднее по дням":0.34989,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":2,"features":251,"volume":38430030000.0,"absolute_error":12760520000.0,"signed_error":-3520034.0,"seconds":15.394641},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322881,"Relative Bias":-0.000918,"|Relative Bias|":0.000918,"метрика конкурса":0.3238,"среднее по дням":0.366058,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":2,"features":251,"volume":2089553000.0,"absolute_error":674677300.0,"signed_error":-1919091.0,"seconds":15.405857},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.330687,"Relative Bias":-0.000199,"|Relative Bias|":0.000199,"метрика конкурса":0.330886,"среднее по дням":0.347487,"variant":"direct","experiment":"baseline","horizon":4,"fold":3,"features":394,"volume":47967270000.0,"absolute_error":15862160000.0,"signed_error":-9557603.0,"seconds":40.972777},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.327965,"Relative Bias":-0.019218,"|Relative Bias|":0.019218,"метрика конкурса":0.347183,"среднее по дням":0.360601,"variant":"direct","experiment":"baseline","horizon":4,"fold":3,"features":394,"volume":2105647000.0,"absolute_error":690578600.0,"signed_error":-40466010.0,"seconds":40.987689},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.331266,"Relative Bias":-0.000119,"|Relative Bias|":0.000119,"метрика конкурса":0.331385,"среднее по дням":0.348176,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":3,"features":238,"volume":47967270000.0,"absolute_error":15889930000.0,"signed_error":-5706366.0,"seconds":18.579393},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.328111,"Relative Bias":-0.017037,"|Relative Bias|":0.017037,"метрика конкурса":0.345148,"среднее по дням":0.360286,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":3,"features":238,"volume":2105647000.0,"absolute_error":690886100.0,"signed_error":-35873620.0,"seconds":18.59097},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.330343,"Relative Bias":-0.000157,"|Relative Bias|":0.000157,"метрика конкурса":0.3305,"среднее по дням":0.348745,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":3,"features":251,"volume":47967270000.0,"absolute_error":15845640000.0,"signed_error":-7538752.0,"seconds":17.969319},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32734,"Relative Bias":-0.015857,"|Relative Bias|":0.015857,"метрика конкурса":0.343196,"среднее по дням":0.356807,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":3,"features":251,"volume":2105647000.0,"absolute_error":689261900.0,"signed_error":-33388370.0,"seconds":17.980823},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.328749,"Relative Bias":-0.000137,"|Relative Bias|":0.000137,"метрика конкурса":0.328886,"среднее по дням":0.345372,"variant":"direct","experiment":"baseline","horizon":4,"fold":4,"features":394,"volume":58105750000.0,"absolute_error":19102200000.0,"signed_error":-7961093.0,"seconds":48.408762},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330256,"Relative Bias":-0.052122,"|Relative Bias|":0.052122,"метрика конкурса":0.382378,"среднее по дням":0.381932,"variant":"direct","experiment":"baseline","horizon":4,"fold":4,"features":394,"volume":2132154000.0,"absolute_error":704155700.0,"signed_error":-111132800.0,"seconds":48.423709},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.329356,"Relative Bias":-0.000218,"|Relative Bias|":0.000218,"метрика конкурса":0.329574,"среднее по дням":0.346113,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":4,"features":238,"volume":58105750000.0,"absolute_error":19137490000.0,"signed_error":-12638490.0,"seconds":21.87173},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.329959,"Relative Bias":-0.048066,"|Relative Bias|":0.048066,"метрика конкурса":0.378024,"среднее по дням":0.377584,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":4,"features":238,"volume":2132154000.0,"absolute_error":703523100.0,"signed_error":-102483100.0,"seconds":21.884332},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.328717,"Relative Bias":-0.000119,"|Relative Bias|":0.000119,"метрика конкурса":0.328836,"среднее по дням":0.34678,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":4,"features":251,"volume":58105750000.0,"absolute_error":19100350000.0,"signed_error":-6916446.0,"seconds":20.624376},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.329311,"Relative Bias":-0.045102,"|Relative Bias|":0.045102,"метрика конкурса":0.374412,"среднее по дням":0.373991,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":4,"features":251,"volume":2132154000.0,"absolute_error":702140600.0,"signed_error":-96164110.0,"seconds":20.635723},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.328278,"Relative Bias":-0.000188,"|Relative Bias|":0.000188,"метрика конкурса":0.328466,"среднее по дням":0.345792,"variant":"direct","experiment":"baseline","horizon":4,"fold":5,"features":394,"volume":68591700000.0,"absolute_error":22517160000.0,"signed_error":-12901090.0,"seconds":57.622364},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322674,"Relative Bias":-0.027637,"|Relative Bias|":0.027637,"метрика конкурса":0.350311,"среднее по дням":0.362621,"variant":"direct","experiment":"baseline","horizon":4,"fold":5,"features":394,"volume":2188764000.0,"absolute_error":706256800.0,"signed_error":-60490790.0,"seconds":57.636844},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.32879,"Relative Bias":-0.000147,"|Relative Bias|":0.000147,"метрика конкурса":0.328937,"среднее по дням":0.3463,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":5,"features":238,"volume":68591700000.0,"absolute_error":22552240000.0,"signed_error":-10089080.0,"seconds":24.121213},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32303,"Relative Bias":-0.025132,"|Relative Bias|":0.025132,"метрика конкурса":0.348162,"среднее по дням":0.361706,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":5,"features":238,"volume":2188764000.0,"absolute_error":707036700.0,"signed_error":-55008550.0,"seconds":24.132155},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.32832,"Relative Bias":-0.000203,"|Relative Bias|":0.000203,"метрика конкурса":0.328523,"среднее по дням":0.346933,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":5,"features":251,"volume":68591700000.0,"absolute_error":22520000000.0,"signed_error":-13920210.0,"seconds":23.863756},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32207,"Relative Bias":-0.023623,"|Relative Bias|":0.023623,"метрика конкурса":0.345693,"среднее по дням":0.356985,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":5,"features":251,"volume":2188764000.0,"absolute_error":704936100.0,"signed_error":-51704360.0,"seconds":23.875444},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.328558,"Relative Bias":-0.000171,"|Relative Bias|":0.000171,"метрика конкурса":0.328729,"среднее по дням":0.34732,"variant":"direct","experiment":"baseline","horizon":4,"fold":6,"features":394,"volume":79044200000.0,"absolute_error":25970620000.0,"signed_error":-13490470.0,"seconds":66.902665},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.336899,"Relative Bias":-0.001986,"|Relative Bias|":0.001986,"метрика конкурса":0.338885,"среднее по дням":0.355824,"variant":"direct","experiment":"baseline","horizon":4,"fold":6,"features":394,"volume":2123980000.0,"absolute_error":715566100.0,"signed_error":-4219089.0,"seconds":66.917824},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.329045,"Relative Bias":-0.000162,"|Relative Bias|":0.000162,"метрика конкурса":0.329207,"среднее по дням":0.347721,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":6,"features":238,"volume":79044200000.0,"absolute_error":26009110000.0,"signed_error":-12779620.0,"seconds":26.983708},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.336657,"Relative Bias":0.000148,"|Relative Bias|":0.000148,"метрика конкурса":0.336805,"среднее по дням":0.35407,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":6,"features":238,"volume":2123980000.0,"absolute_error":715053100.0,"signed_error":313635.5,"seconds":26.994465},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.328768,"Relative Bias":-0.000126,"|Relative Bias|":0.000126,"метрика конкурса":0.328894,"среднее по дням":0.348498,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":6,"features":251,"volume":79044200000.0,"absolute_error":25987210000.0,"signed_error":-9986167.0,"seconds":27.998198},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.335182,"Relative Bias":-0.003856,"|Relative Bias|":0.003856,"метрика конкурса":0.339038,"среднее по дням":0.354363,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":6,"features":251,"volume":2123980000.0,"absolute_error":711919200.0,"signed_error":-8190124.0,"seconds":28.009337},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.325223,"Relative Bias":-0.000401,"|Relative Bias|":0.000401,"метрика конкурса":0.325624,"среднее по дням":0.343281,"variant":"direct","experiment":"baseline","horizon":5,"fold":1,"features":396,"volume":28982240000.0,"absolute_error":9425680000.0,"signed_error":-11620940.0,"seconds":54.759062},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326673,"Relative Bias":-0.002819,"|Relative Bias|":0.002819,"метрика конкурса":0.329492,"среднее по дням":0.339757,"variant":"direct","experiment":"baseline","horizon":5,"fold":1,"features":396,"volume":1895322000.0,"absolute_error":619151200.0,"signed_error":-5342510.0,"seconds":54.774553},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.326751,"Relative Bias":-0.000468,"|Relative Bias|":0.000468,"метрика конкурса":0.327218,"среднее по дням":0.34459,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":1,"features":240,"volume":28982240000.0,"absolute_error":9469967000.0,"signed_error":-13557140.0,"seconds":26.571487},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326877,"Relative Bias":-0.001902,"|Relative Bias|":0.001902,"метрика конкурса":0.328779,"среднее по дням":0.338596,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":1,"features":240,"volume":1895322000.0,"absolute_error":619537200.0,"signed_error":-3604476.0,"seconds":26.58236},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.32647,"Relative Bias":-0.000384,"|Relative Bias|":0.000384,"метрика конкурса":0.326854,"среднее по дням":0.346847,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":1,"features":253,"volume":28982240000.0,"absolute_error":9461838000.0,"signed_error":-11118120.0,"seconds":27.468946},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326679,"Relative Bias":-0.003152,"|Relative Bias|":0.003152,"метрика конкурса":0.329831,"среднее по дням":0.342278,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":1,"features":253,"volume":1895322000.0,"absolute_error":619162000.0,"signed_error":-5974875.0,"seconds":27.481195},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.324645,"Relative Bias":-0.000327,"|Relative Bias|":0.000327,"метрика конкурса":0.324971,"среднее по дням":0.342208,"variant":"direct","experiment":"baseline","horizon":5,"fold":2,"features":396,"volume":37770830000.0,"absolute_error":12262090000.0,"signed_error":-12335590.0,"seconds":68.901301},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.327693,"Relative Bias":0.004008,"|Relative Bias|":0.004008,"метрика конкурса":0.331701,"среднее по дням":0.353419,"variant":"direct","experiment":"baseline","horizon":5,"fold":2,"features":396,"volume":2037994000.0,"absolute_error":667837000.0,"signed_error":8168032.0,"seconds":68.917342},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3258925602,"Relative Bias":-0.0002473606,"|Relative Bias|":0.0002473606,"метрика конкурса":0.3261399208,"среднее по дням":0.3433559592,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":2,"features":240,"volume":37770825728.0,"absolute_error":12309230870.45662117,"signed_error":-9343013.3269762993,"seconds":48.973573159},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3275485661,"Relative Bias":0.0032071498,"|Relative Bias|":0.0032071498,"метрика конкурса":0.3307557159,"среднее по дням":0.3542766583,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":2,"features":240,"volume":2037994240.0,"absolute_error":667542085.1868829727,"signed_error":6536152.7266530991,"seconds":48.991571809},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.325556901,"Relative Bias":-0.0002175257,"|Relative Bias|":0.0002175257,"метрика конкурса":0.3257744267,"среднее по дням":0.3439150523,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":2,"features":253,"volume":37770825728.0,"absolute_error":12296552747.6319446564,"signed_error":-8216125.6448960304,"seconds":33.390384402},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3261061237,"Relative Bias":-0.0006730483,"|Relative Bias|":0.0006730483,"метрика конкурса":0.326779172,"среднее по дням":0.3507051792,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":2,"features":253,"volume":2037994240.0,"absolute_error":664602395.8430929184,"signed_error":-1371668.5709924698,"seconds":33.404252489},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3220745552,"Relative Bias":-0.0003219629,"|Relative Bias|":0.0003219629,"метрика конкурса":0.3223965181,"среднее по дням":0.3396629673,"variant":"direct","experiment":"baseline","horizon":5,"fold":3,"features":396,"volume":47295012864.0,"absolute_error":15232520385.181640625,"signed_error":-15227238.7280902863,"seconds":82.17290469},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3338893972,"Relative Bias":-0.0162495889,"|Relative Bias|":0.0162495889,"метрика конкурса":0.3501389861,"среднее по дням":0.3565312493,"variant":"direct","experiment":"baseline","horizon":5,"fold":3,"features":396,"volume":2092325120.0,"absolute_error":698605137.7212486267,"signed_error":-33999421.3629379272,"seconds":82.199701691},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3232982088,"Relative Bias":-0.0002748541,"|Relative Bias|":0.0002748541,"метрика конкурса":0.3235730629,"среднее по дням":0.3407974947,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":3,"features":240,"volume":47295012864.0,"absolute_error":15290393099.6034355164,"signed_error":-12999228.6741576195,"seconds":41.017133954},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3339902086,"Relative Bias":-0.0131360842,"|Relative Bias|":0.0131360842,"метрика конкурса":0.3471262928,"среднее по дням":0.3551184259,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":3,"features":240,"volume":2092325120.0,"absolute_error":698816067.9588947296,"signed_error":-27484957.4554576874,"seconds":41.02934219},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3233904558,"Relative Bias":-0.0003375252,"|Relative Bias|":0.0003375252,"метрика конкурса":0.3237279811,"среднее по дням":0.3420538083,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":3,"features":253,"volume":47295012864.0,"absolute_error":15294755924.5813331604,"signed_error":-15963259.9472842216,"seconds":42.008203961},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3335609506,"Relative Bias":-0.0152144669,"|Relative Bias|":0.0152144669,"метрика конкурса":0.3487754175,"среднее по дням":0.3570332781,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":3,"features":253,"volume":2092325120.0,"absolute_error":697917920.7247457504,"signed_error":-31833609.6484079361,"seconds":42.020966404},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.319907896,"Relative Bias":-0.000372911,"|Relative Bias|":0.000372911,"метрика конкурса":0.320280807,"среднее по дням":0.338324632,"variant":"direct","experiment":"baseline","horizon":5,"fold":4,"features":396,"volume":57567428608.0,"absolute_error":18416275298.4614944458,"signed_error":-21467525.509262085,"seconds":96.046572376},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3330326143,"Relative Bias":-0.0369698758,"|Relative Bias|":0.0369698758,"метрика конкурса":0.3700024901,"среднее по дням":0.3694943782,"variant":"direct","experiment":"baseline","horizon":5,"fold":4,"features":396,"volume":2099699968.0,"absolute_error":699268582.2535533905,"signed_error":-77625648.3890590668,"seconds":96.06316842},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.321049124,"Relative Bias":-0.0003071626,"|Relative Bias|":0.0003071626,"метрика конкурса":0.3213562866,"среднее по дням":0.3395510298,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":4,"features":240,"volume":57567428608.0,"absolute_error":18481972860.0100212097,"signed_error":-17682562.8044843674,"seconds":49.029444997},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3326066369,"Relative Bias":-0.032284866,"|Relative Bias|":0.032284866,"метрика конкурса":0.3648915029,"среднее по дням":0.3644073419,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":4,"features":240,"volume":2099699968.0,"absolute_error":698374157.4868202209,"signed_error":-67788533.4112224579,"seconds":49.0412673},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3215140337,"Relative Bias":-0.0001834831,"|Relative Bias|":0.0001834831,"метрика конкурса":0.3216975168,"среднее по дням":0.34087996,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":4,"features":253,"volume":57567428608.0,"absolute_error":18508736518.1576309204,"signed_error":-10562648.6900091171,"seconds":49.419136091},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3315196278,"Relative Bias":-0.0322630199,"|Relative Bias|":0.0322630199,"метрика конкурса":0.3637826476,"среднее по дням":0.3653935491,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":4,"features":253,"volume":2099699968.0,"absolute_error":696091764.4118471146,"signed_error":-67742662.996846199,"seconds":49.432658235},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3197474575,"Relative Bias":-0.0003172375,"|Relative Bias|":0.0003172375,"метрика конкурса":0.320064695,"среднее по дням":0.3381908176,"variant":"direct","experiment":"baseline","horizon":5,"fold":5,"features":396,"volume":68053016576.0,"absolute_error":21759778826.0302505493,"signed_error":-21588969.7288570404,"seconds":110.7364617},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3263679578,"Relative Bias":-0.0126716846,"|Relative Bias|":0.0126716846,"метрика конкурса":0.3390396424,"среднее по дням":0.3463513778,"variant":"direct","experiment":"baseline","horizon":5,"fold":5,"features":396,"volume":2166344704.0,"absolute_error":707025503.5157775879,"signed_error":-27451237.0829811096,"seconds":110.753158839},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3206190826,"Relative Bias":-0.0003178584,"|Relative Bias|":0.0003178584,"метрика конкурса":0.3209369411,"среднее по дням":0.339060004,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":5,"features":240,"volume":68053016576.0,"absolute_error":21819095545.4380302429,"signed_error":-21631225.3092823029,"seconds":57.42628676},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3263522878,"Relative Bias":-0.0107743457,"|Relative Bias|":0.0107743457,"метрика конкурса":0.3371266335,"среднее по дням":0.3448282315,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":5,"features":240,"volume":2166344704.0,"absolute_error":706991556.9041757584,"signed_error":-23340946.8685617447,"seconds":57.438308624},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3214948539,"Relative Bias":-0.0002018624,"|Relative Bias|":0.0002018624,"метрика конкурса":0.3216967163,"среднее по дням":0.3408447904,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":5,"features":253,"volume":68053016576.0,"absolute_error":21878694424.0274009705,"signed_error":-13737346.1802897453,"seconds":57.404605952},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32651859,"Relative Bias":-0.0126243534,"|Relative Bias|":0.0126243534,"метрика конкурса":0.3391429434,"среднее по дням":0.3461383234,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":5,"features":253,"volume":2166344704.0,"absolute_error":707351824.8037128448,"signed_error":-27348701.382183075,"seconds":57.418568484},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3203474163,"Relative Bias":-0.0004086869,"|Relative Bias|":0.0004086869,"метрика конкурса":0.3207561032,"среднее по дням":0.3391503401,"variant":"direct","experiment":"baseline","horizon":5,"fold":6,"features":396,"volume":78463197184.0,"absolute_error":25135482911.2231254578,"signed_error":-32066878.0491724014,"seconds":126.726696977},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3355475543,"Relative Bias":0.006190355,"|Relative Bias|":0.006190355,"метрика конкурса":0.3417379092,"среднее по дням":0.3684425998,"variant":"direct","experiment":"baseline","horizon":5,"fold":6,"features":396,"volume":2098418688.0,"absolute_error":704119281.0512266159,"signed_error":12989957.0085954666,"seconds":126.74306855},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3213472316,"Relative Bias":-0.0004024068,"|Relative Bias|":0.0004024068,"метрика конкурса":0.3217496383,"среднее по дням":0.3402537272,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":6,"features":240,"volume":78463197184.0,"absolute_error":25213931613.852771759,"signed_error":-31574120.8897008896,"seconds":65.518742652},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3358728342,"Relative Bias":0.0086983873,"|Relative Bias|":0.0086983873,"метрика конкурса":0.3445712215,"среднее по дням":0.3674845739,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":6,"features":240,"volume":2098418688.0,"absolute_error":704801854.5317678452,"signed_error":18252858.9942407608,"seconds":65.530299041},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3221642799,"Relative Bias":-0.0002991446,"|Relative Bias|":0.0002991446,"метрика конкурса":0.3224634246,"среднее по дням":0.3419720954,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":6,"features":253,"volume":78463197184.0,"absolute_error":25278039843.5184631348,"signed_error":-23471843.1488647461,"seconds":66.449957065},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3347779889,"Relative Bias":0.0043020649,"|Relative Bias|":0.0043020649,"метрика конкурса":0.3390800538,"среднее по дням":0.3642895036,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":6,"features":253,"volume":2098418688.0,"absolute_error":702504410.7724237442,"signed_error":9027533.5946035385,"seconds":66.462881557},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3290617682,"Relative Bias":-0.0001610266,"|Relative Bias|":0.0001610266,"метрика конкурса":0.3292227948,"среднее по дням":0.3493864269,"variant":"direct","experiment":"baseline","horizon":6,"fold":1,"features":398,"volume":28578764800.0,"absolute_error":9404179031.0473365784,"signed_error":-4601941.9163479805,"seconds":44.872968922},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3349174343,"Relative Bias":-0.0129531533,"|Relative Bias|":0.0129531533,"метрика конкурса":0.3478705876,"среднее по дням":0.3630880011,"variant":"direct","experiment":"baseline","horizon":6,"fold":1,"features":398,"volume":1852648704.0,"absolute_error":620484362.6891727448,"signed_error":-23997643.2030105591,"seconds":44.888642885},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3308346175,"Relative Bias":-0.0000455182,"|Relative Bias|":0.0000455182,"метрика конкурса":0.3308801357,"среднее по дням":0.3514818842,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":1,"features":242,"volume":28578764800.0,"absolute_error":9454844875.4594154358,"signed_error":-1300854.2940034866,"seconds":22.059835017},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3344794565,"Relative Bias":-0.0087621455,"|Relative Bias|":0.0087621455,"метрика конкурса":0.343241602,"среднее по дням":0.3598431226,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":1,"features":242,"volume":1852648704.0,"absolute_error":619672943.5813188553,"signed_error":-16233177.8679990768,"seconds":22.071846745},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3312649192,"Relative Bias":-0.0000590498,"|Relative Bias|":0.0000590498,"метрика конкурса":0.3313239691,"среднее по дням":0.3546214464,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":1,"features":255,"volume":28578764800.0,"absolute_error":9467142367.4332408905,"signed_error":-1687571.5091838837,"seconds":21.466515311},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3345277276,"Relative Bias":-0.0139055866,"|Relative Bias|":0.0139055866,"метрика конкурса":0.3484333142,"среднее по дням":0.3621558604,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":1,"features":255,"volume":1852648704.0,"absolute_error":619762372.9919481277,"signed_error":-25762167.4691343307,"seconds":21.479100526},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3287881312,"Relative Bias":-0.0001606251,"|Relative Bias|":0.0001606251,"метрика конкурса":0.3289487562,"среднее по дням":0.3488794087,"variant":"direct","experiment":"baseline","horizon":6,"fold":2,"features":398,"volume":37203214336.0,"absolute_error":12231974882.604095459,"signed_error":-5975768.5513687134,"seconds":55.316608345},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3230174958,"Relative Bias":-0.010436163,"|Relative Bias|":0.010436163,"метрика конкурса":0.3334536588,"среднее по дням":0.3533652483,"variant":"direct","experiment":"baseline","horizon":6,"fold":2,"features":398,"volume":2012951040.0,"absolute_error":650218431.8666467667,"signed_error":-21007486.0927591324,"seconds":55.33354801},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3301649442,"Relative Bias":-0.0001437028,"|Relative Bias|":0.0001437028,"метрика конкурса":0.330308647,"среднее по дням":0.3504479484,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":2,"features":242,"volume":37203214336.0,"absolute_error":12283196752.5314064026,"signed_error":-5346205.9469413757,"seconds":27.481508152},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3226464396,"Relative Bias":-0.0089806957,"|Relative Bias|":0.0089806957,"метрика конкурса":0.3316271353,"среднее по дням":0.3516910753,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":2,"features":242,"volume":2012951040.0,"absolute_error":649471513.9462585449,"signed_error":-18077701.4342079163,"seconds":27.493262425},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.330489709,"Relative Bias":-0.0000525442,"|Relative Bias|":0.0000525442,"метрика конкурса":0.3305422532,"среднее по дням":0.352023933,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":2,"features":255,"volume":37203214336.0,"absolute_error":12295279044.2358283997,"signed_error":-1954812.1349406242,"seconds":27.286204245},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3216708971,"Relative Bias":-0.0161642934,"|Relative Bias|":0.0161642934,"метрика конкурса":0.3378351905,"среднее по дням":0.3505385027,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":2,"features":255,"volume":2012951040.0,"absolute_error":647507794.4646615982,"signed_error":-32537932.6936731339,"seconds":27.299035495},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3252515825,"Relative Bias":-0.0002919295,"|Relative Bias|":0.0002919295,"метрика конкурса":0.325543512,"среднее по дням":0.344798595,"variant":"direct","experiment":"baseline","horizon":6,"fold":3,"features":398,"volume":46733324288.0,"absolute_error":15200087243.0560722351,"signed_error":-13642834.3713612556,"seconds":66.839284474},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3310966509,"Relative Bias":-0.0111023084,"|Relative Bias|":0.0111023084,"метрика конкурса":0.3421989593,"среднее по дням":0.3733692579,"variant":"direct","experiment":"baseline","horizon":6,"fold":3,"features":398,"volume":2067840896.0,"absolute_error":684655201.5443620682,"signed_error":-22957807.6268815994,"seconds":66.856150473},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3265494749,"Relative Bias":-0.0002832558,"|Relative Bias|":0.0002832558,"метрика конкурса":0.3268327308,"среднее по дням":0.3463476492,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":3,"features":242,"volume":46733324288.0,"absolute_error":15260742069.1313972473,"signed_error":-13237487.0136709213,"seconds":33.126324178},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3317657441,"Relative Bias":-0.0067544354,"|Relative Bias|":0.0067544354,"метрика конкурса":0.3385201795,"среднее по дням":0.3745046457,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":3,"features":242,"volume":2067840896.0,"absolute_error":686038779.9097824097,"signed_error":-13967097.8899383545,"seconds":33.138329321},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3271069499,"Relative Bias":-0.0002415714,"|Relative Bias|":0.0002415714,"метрика конкурса":0.3273485213,"среднее по дням":0.3484104887,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":3,"features":255,"volume":46733324288.0,"absolute_error":15286794726.5028266907,"signed_error":-11289436.2449975014,"seconds":33.813896691},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3319304685,"Relative Bias":-0.0113037478,"|Relative Bias|":0.0113037478,"метрика конкурса":0.3432342163,"среднее по дням":0.3761951947,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":3,"features":255,"volume":2067840896.0,"absolute_error":686379403.6506595612,"signed_error":-23374352.1745586395,"seconds":33.827019508},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3226165822,"Relative Bias":-0.0002088089,"|Relative Bias|":0.0002088089,"метрика конкурса":0.3228253911,"среднее по дням":0.3433704481,"variant":"direct","experiment":"baseline","horizon":6,"fold":4,"features":398,"volume":57034534912.0,"absolute_error":18400287505.2232284546,"signed_error":-11909318.7453927994,"seconds":79.005512622},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3411939852,"Relative Bias":-0.0156446168,"|Relative Bias|":0.0156446168,"метрика конкурса":0.356838602,"среднее по дням":0.3723101009,"variant":"direct","experiment":"baseline","horizon":6,"fold":4,"features":398,"volume":2043491328.0,"absolute_error":697226960.9139795303,"signed_error":-31969639.2419195175,"seconds":79.022702547},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3236939176,"Relative Bias":-0.0001799987,"|Relative Bias|":0.0001799987,"метрика конкурса":0.3238739164,"среднее по дням":0.344616169,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":4,"features":242,"volume":57034534912.0,"absolute_error":18461732831.8701057434,"signed_error":-10266145.2817502022,"seconds":39.282792816},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.34132387,"Relative Bias":-0.0093242146,"|Relative Bias|":0.0093242146,"метрика конкурса":0.3506480846,"среднее по дням":0.3711121511,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":4,"features":242,"volume":2043491328.0,"absolute_error":697492379.367609024,"signed_error":-19053951.8805599213,"seconds":39.295297491},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3243880266,"Relative Bias":-0.0000813823,"|Relative Bias|":0.0000813823,"метрика конкурса":0.3244694089,"среднее по дням":0.3468440975,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":4,"features":255,"volume":57034534912.0,"absolute_error":18501321018.2074775696,"signed_error":-4641600.7163705826,"seconds":40.312742894},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3407896229,"Relative Bias":-0.0112288162,"|Relative Bias|":0.0112288162,"метрика конкурса":0.352018439,"среднее по дням":0.3676852662,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":4,"features":255,"volume":2043491328.0,"absolute_error":696400649.8785095215,"signed_error":-22945988.849029541,"seconds":40.325830261},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3217616264,"Relative Bias":-0.0003369904,"|Relative Bias|":0.0003369904,"метрика конкурса":0.3220986167,"среднее по дням":0.3427272928,"variant":"direct","experiment":"baseline","horizon":6,"fold":5,"features":398,"volume":67535437824.0,"absolute_error":21730313934.4860229492,"signed_error":-22758794.4905920029,"seconds":90.931665109},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3416043485,"Relative Bias":0.0214799828,"|Relative Bias|":0.0214799828,"метрика конкурса":0.3630843313,"среднее по дням":0.3908181606,"variant":"direct","experiment":"baseline","horizon":6,"fold":5,"features":398,"volume":2081811584.0,"absolute_error":711155899.3936805725,"signed_error":44717277.5428180695,"seconds":90.948286704},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3227619587,"Relative Bias":-0.0002857249,"|Relative Bias|":0.0002857249,"метрика конкурса":0.3230476836,"среднее по дням":0.3439102943,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":5,"features":242,"volume":67535437824.0,"absolute_error":21797871818.5621604919,"signed_error":-19296558.7328834534,"seconds":46.084708426},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3419335916,"Relative Bias":0.0258501884,"|Relative Bias|":0.0258501884,"метрика конкурса":0.3677837799,"среднее по дням":0.3916841507,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":5,"features":242,"volume":2081811584.0,"absolute_error":711841321.4763040543,"signed_error":53815222.300160408,"seconds":46.096734917},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3234476511,"Relative Bias":-0.0002739907,"|Relative Bias|":0.0002739907,"метрика конкурса":0.3237216418,"среднее по дням":0.3464504602,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":5,"features":255,"volume":67535437824.0,"absolute_error":21844180361.2865180969,"signed_error":-18504085.4807658195,"seconds":46.371296288},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.341599012,"Relative Bias":0.0238859438,"|Relative Bias|":0.0238859438,"метрика конкурса":0.3654849558,"среднее по дням":0.3902549981,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":5,"features":255,"volume":2081811584.0,"absolute_error":711144789.8667421341,"signed_error":49726035.08340168,"seconds":46.384454274},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3222435948,"Relative Bias":-0.0002621299,"|Relative Bias|":0.0002621299,"метрика конкурса":0.3225057248,"среднее по дням":0.3430046621,"variant":"direct","experiment":"baseline","horizon":6,"fold":6,"features":398,"volume":77883179008.0,"absolute_error":25097358588.1725349426,"signed_error":-20415515.235748291,"seconds":103.296915707},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3450175903,"Relative Bias":0.0214193863,"|Relative Bias|":0.0214193863,"метрика конкурса":0.3664369767,"среднее по дням":0.3692536568,"variant":"direct","experiment":"baseline","horizon":6,"fold":6,"features":398,"volume":2046648320.0,"absolute_error":706129685.053144455,"signed_error":43837951.9111785889,"seconds":103.313569729},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3232323851,"Relative Bias":-0.0002536848,"|Relative Bias|":0.0002536848,"метрика конкурса":0.3234860699,"среднее по дням":0.3441157707,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":6,"features":242,"volume":77883179008.0,"absolute_error":25174368729.8835525513,"signed_error":-19757777.8525018692,"seconds":53.339179438},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.345868063,"Relative Bias":0.0255014946,"|Relative Bias|":0.0255014946,"метрика конкурса":0.3713695576,"среднее по дням":0.373056165,"variant":"direct","experiment":"thin_offsets","horizon":6,"fold":6,"features":242,"volume":2046648320.0,"absolute_error":707870303.5944166183,"signed_error":52192592.0344266891,"seconds":53.350442256},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3240247471,"Relative Bias":-0.000180897,"|Relative Bias|":0.000180897,"метрика конкурса":0.3242056442,"среднее по дням":0.3466016482,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":6,"features":255,"volume":77883179008.0,"absolute_error":25236080408.3834457397,"signed_error":-14088838.3991384506,"seconds":52.917457638},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3436689205,"Relative Bias":0.0211656622,"|Relative Bias|":0.0211656622,"метрика конкурса":0.3648345827,"среднее по дням":0.3667235582,"variant":"direct","experiment":"compact_with_additions","horizon":6,"fold":6,"features":255,"volume":2046648320.0,"absolute_error":703369432.165345192,"signed_error":43318667.8207788467,"seconds":52.930335264},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.330238522,"Relative Bias":-0.0003802372,"|Relative Bias|":0.0003802372,"метрика конкурса":0.3306187592,"среднее по дням":0.3501050867,"variant":"direct","experiment":"baseline","horizon":7,"fold":1,"features":400,"volume":28152850432.0,"absolute_error":9297155817.2304611206,"signed_error":-10704760.6903896332,"seconds":52.19638056},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3350548764,"Relative Bias":-0.0399616183,"|Relative Bias|":0.0399616183,"метрика конкурса":0.3750164947,"среднее по дням":0.3739670627,"variant":"direct","experiment":"baseline","horizon":7,"fold":1,"features":400,"volume":1902749312.0,"absolute_error":637525428.4851198196,"signed_error":-76036940.8371362686,"seconds":52.212262178},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3317536166,"Relative Bias":-0.0004205504,"|Relative Bias|":0.0004205504,"метрика конкурса":0.332174167,"среднее по дням":0.3519188698,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":1,"features":244,"volume":28152850432.0,"absolute_error":9339810048.7513427734,"signed_error":-11839693.9609036446,"seconds":27.007800161},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3348802095,"Relative Bias":-0.0401233798,"|Relative Bias|":0.0401233798,"метрика конкурса":0.3750035894,"среднее по дням":0.3740056641,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":1,"features":244,"volume":1902749312.0,"absolute_error":637193081.2730464935,"signed_error":-76344732.5202140808,"seconds":27.019733096},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3330831625,"Relative Bias":-0.0002865135,"|Relative Bias|":0.0002865135,"метрика конкурса":0.333369676,"среднее по дням":0.354481884,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":1,"features":257,"volume":28152850432.0,"absolute_error":9377240557.1314888,"signed_error":-8066172.1411929131,"seconds":27.171755546},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3342148938,"Relative Bias":-0.0449046156,"|Relative Bias|":0.0449046156,"метрика конкурса":0.3791195094,"среднее по дням":0.3780081444,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":1,"features":257,"volume":1902749312.0,"absolute_error":635927152.2654294968,"signed_error":-85442225.4922304153,"seconds":27.184175781},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.329652414,"Relative Bias":-0.0003843709,"|Relative Bias|":0.0003843709,"метрика конкурса":0.3300367849,"среднее по дням":0.3487673605,"variant":"direct","experiment":"baseline","horizon":7,"fold":2,"features":400,"volume":36707352576.0,"absolute_error":12100668291.6801071167,"signed_error":-14109237.7171411514,"seconds":66.785540195},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3335939773,"Relative Bias":-0.0426908902,"|Relative Bias|":0.0426908902,"метрика конкурса":0.3762848676,"среднее по дням":0.374433209,"variant":"direct","experiment":"baseline","horizon":7,"fold":2,"features":400,"volume":2067504000.0,"absolute_error":689706870.5349025726,"signed_error":-88263584.7678318024,"seconds":66.802264457},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3310429956,"Relative Bias":-0.0003542827,"|Relative Bias|":0.0003542827,"метрика конкурса":0.3313972783,"среднее по дням":0.3505597724,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":2,"features":244,"volume":36707352576.0,"absolute_error":12151712862.2304611206,"signed_error":-13004781.459321022,"seconds":34.015578514},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3338808219,"Relative Bias":-0.0383891944,"|Relative Bias|":0.0383891944,"метрика конкурса":0.3722700163,"среднее по дням":0.3715291248,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":2,"features":244,"volume":2067504000.0,"absolute_error":690299922.8649673462,"signed_error":-79369811.5875854492,"seconds":34.027592597},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3321619075,"Relative Bias":-0.0002759793,"|Relative Bias|":0.0002759793,"метрика конкурса":0.3324378868,"среднее по дням":0.3530559779,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":2,"features":257,"volume":36707352576.0,"absolute_error":12192785160.9514560699,"signed_error":-10130469.8423042297,"seconds":34.901959009},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3318348939,"Relative Bias":-0.0463099747,"|Relative Bias|":0.0463099747,"метрика конкурса":0.3781448686,"среднее по дням":0.3762839824,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":2,"features":257,"volume":2067504000.0,"absolute_error":686069958.4936485291,"signed_error":-95746056.2815799713,"seconds":34.914877206},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3264825043,"Relative Bias":-0.0003267433,"|Relative Bias|":0.0003267433,"метрика конкурса":0.3268092476,"среднее по дням":0.3457665145,"variant":"direct","experiment":"baseline","horizon":7,"fold":3,"features":400,"volume":46176509952.0,"absolute_error":15075822346.7949352264,"signed_error":-15087865.6156349182,"seconds":80.017682893},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3292462417,"Relative Bias":-0.0239281976,"|Relative Bias|":0.0239281976,"метрика конкурса":0.3531744393,"среднее по дням":0.3637621956,"variant":"direct","experiment":"baseline","horizon":7,"fold":3,"features":400,"volume":2101351552.0,"absolute_error":691862131.9673147202,"signed_error":-50281557.3661050797,"seconds":80.034195964},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3276498484,"Relative Bias":-0.0002168,"|Relative Bias|":0.0002168,"метрика конкурса":0.3278666484,"среднее по дням":0.3472113431,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":3,"features":244,"volume":46176509952.0,"absolute_error":15129726223.2887573242,"signed_error":-10011067.1125340462,"seconds":41.706219905},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3295217988,"Relative Bias":-0.0197256101,"|Relative Bias|":0.0197256101,"метрика конкурса":0.3492474089,"среднее по дням":0.3628062965,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":3,"features":244,"volume":2101351552.0,"absolute_error":692441174.3049564362,"signed_error":-41450443.219991684,"seconds":41.718301607},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3289859084,"Relative Bias":-0.0001645124,"|Relative Bias|":0.0001645124,"метрика конкурса":0.3291504208,"среднее по дням":0.3499520685,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":3,"features":257,"volume":46176509952.0,"absolute_error":15191420810.2164821625,"signed_error":-7596606.933324337,"seconds":42.329158309},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3292403748,"Relative Bias":-0.0235066306,"|Relative Bias|":0.0235066306,"метрика конкурса":0.3527470053,"среднее по дням":0.3635458353,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":3,"features":257,"volume":2101351552.0,"absolute_error":691849803.4369268417,"signed_error":-49395696.857424736,"seconds":42.341456355},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3235244819,"Relative Bias":-0.0003429381,"|Relative Bias|":0.0003429381,"метрика конкурса":0.3238674201,"среднее по дням":0.3434831139,"variant":"direct","experiment":"baseline","horizon":7,"fold":4,"features":400,"volume":56473100288.0,"absolute_error":18270430282.3749237061,"signed_error":-19366778.533121109,"seconds":95.202321324},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3454460907,"Relative Bias":0.0018541289,"|Relative Bias|":0.0018541289,"метрика конкурса":0.3473002196,"среднее по дням":0.3751841876,"variant":"direct","experiment":"baseline","horizon":7,"fold":4,"features":400,"volume":2014791168.0,"absolute_error":696001723.8590888977,"signed_error":3735682.5650997162,"seconds":95.21889604},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3245704449,"Relative Bias":-0.0001907109,"|Relative Bias|":0.0001907109,"метрика конкурса":0.3247611558,"среднее по дням":0.3447635309,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":4,"features":244,"volume":56473100288.0,"absolute_error":18329499053.1094779968,"signed_error":-10770036.949177742,"seconds":50.044791351},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3459592366,"Relative Bias":0.0080201739,"|Relative Bias|":0.0080201739,"метрика конкурса":0.3539794105,"среднее по дням":0.3767503098,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":4,"features":244,"volume":2014791168.0,"absolute_error":697035605.7524642944,"signed_error":16158975.2615756989,"seconds":50.057001491},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3260777918,"Relative Bias":-0.0002004494,"|Relative Bias|":0.0002004494,"метрика конкурса":0.3262782411,"среднее по дням":0.3478093565,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":4,"features":257,"volume":56473100288.0,"absolute_error":18414623601.7305755615,"signed_error":-11319996.3286123276,"seconds":50.130082354},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3450958157,"Relative Bias":0.0079567852,"|Relative Bias|":0.0079567852,"метрика конкурса":0.3530526009,"среднее по дням":0.3713596958,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":4,"features":257,"volume":2014791168.0,"absolute_error":695295992.9049940109,"signed_error":16031260.3683538437,"seconds":50.142817414},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.322464611,"Relative Bias":-0.0003670256,"|Relative Bias|":0.0003670256,"метрика конкурса":0.3228316367,"среднее по дням":0.3428533232,"variant":"direct","experiment":"baseline","horizon":7,"fold":5,"features":400,"volume":66969038848.0,"absolute_error":21595144178.948261261,"signed_error":-24579352.9487190247,"seconds":110.627891887},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3415942521,"Relative Bias":0.023984867,"|Relative Bias|":0.023984867,"метрика конкурса":0.3655791192,"среднее по дням":0.397917619,"variant":"direct","experiment":"baseline","horizon":7,"fold":5,"features":400,"volume":2076204800.0,"absolute_error":709219604.0543107986,"signed_error":49797494.5101156235,"seconds":110.644511685},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3234088253,"Relative Bias":-0.0002924938,"|Relative Bias|":0.0002924938,"метрика конкурса":0.3237013191,"среднее по дням":0.343982207,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":5,"features":244,"volume":66969038848.0,"absolute_error":21658377300.9986114502,"signed_error":-19588025.8328256607,"seconds":57.870617156},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3415209138,"Relative Bias":0.0265112364,"|Relative Bias|":0.0265112364,"метрика конкурса":0.3680321502,"среднее по дням":0.397823409,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":5,"features":244,"volume":2076204800.0,"absolute_error":709067338.6890573502,"signed_error":55042754.5282258987,"seconds":57.88242139},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3246890828,"Relative Bias":-0.0002364422,"|Relative Bias|":0.0002364422,"метрика конкурса":0.324925525,"среднее по дням":0.3472445165,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":5,"features":257,"volume":66969038848.0,"absolute_error":21744114909.2032546997,"signed_error":-15834307.8157839775,"seconds":58.797806699},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3406967071,"Relative Bias":0.0263429681,"|Relative Bias|":0.0263429681,"метрика конкурса":0.3670396752,"среднее по дням":0.3959267933,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":5,"features":257,"volume":2076204800.0,"absolute_error":707356116.7983884811,"signed_error":54693395.188750267,"seconds":58.810148173},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3230063464,"Relative Bias":-0.0002757515,"|Relative Bias|":0.0002757515,"метрика конкурса":0.3232820979,"среднее по дням":0.3432454154,"variant":"direct","experiment":"baseline","horizon":7,"fold":6,"features":400,"volume":77253828608.0,"absolute_error":24953477497.4094924927,"signed_error":-21302859.0405244827,"seconds":126.249497623},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3449278796,"Relative Bias":0.0216036174,"|Relative Bias|":0.0216036174,"метрика конкурса":0.366531497,"среднее по дням":0.3845958157,"variant":"direct","experiment":"baseline","horizon":7,"fold":6,"features":400,"volume":2046924800.0,"absolute_error":706041419.2368545532,"signed_error":44220979.4442977905,"seconds":126.266203019},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3239629267,"Relative Bias":-0.0002378579,"|Relative Bias|":0.0002378579,"метрика конкурса":0.3242007845,"среднее по дням":0.3444262881,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":6,"features":244,"volume":77253828608.0,"absolute_error":25027376984.3082427979,"signed_error":-18375431.8559427261,"seconds":66.113246126},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3451601388,"Relative Bias":0.0242919251,"|Relative Bias|":0.0242919251,"метрика конкурса":0.3694520639,"среднее по дням":0.3857978825,"variant":"direct","experiment":"thin_offsets","horizon":7,"fold":6,"features":244,"volume":2046924800.0,"absolute_error":706516836.3620290756,"signed_error":49723743.1179609299,"seconds":66.125303398},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3252956168,"Relative Bias":-0.0002051112,"|Relative Bias|":0.0002051112,"метрика конкурса":0.325500728,"среднее по дням":0.3471202946,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":6,"features":257,"volume":77253828608.0,"absolute_error":25130332404.5697441101,"signed_error":-15845621.9995279312,"seconds":67.1912757},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.342906168,"Relative Bias":0.0192697263,"|Relative Bias|":0.0192697263,"метрика конкурса":0.3621758943,"среднее по дням":0.3825699428,"variant":"direct","experiment":"compact_with_additions","horizon":7,"fold":6,"features":257,"volume":2046924800.0,"absolute_error":701903127.6898107529,"signed_error":39443680.0312128067,"seconds":67.203636826},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3297722274,"Relative Bias":-0.0004307601,"|Relative Bias|":0.0004307601,"метрика конкурса":0.3302029875,"среднее по дням":0.3478248464,"variant":"direct","experiment":"baseline","horizon":8,"fold":1,"features":402,"volume":27932401664.0,"absolute_error":9211330429.5554904938,"signed_error":-12032165.1378417015,"seconds":52.515772066},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3294691636,"Relative Bias":-0.0488497634,"|Relative Bias|":0.0488497634,"метрика конкурса":0.378318927,"среднее по дням":0.3772469684,"variant":"direct","experiment":"baseline","horizon":8,"fold":1,"features":402,"volume":1937103104.0,"absolute_error":638215736.2294874191,"signed_error":-94627027.8487386703,"seconds":52.532229386},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3312401434,"Relative Bias":-0.0002609938,"|Relative Bias|":0.0002609938,"метрика конкурса":0.3315011372,"среднее по дням":0.3495662848,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":1,"features":246,"volume":27932401664.0,"absolute_error":9252332848.6393432617,"signed_error":-7290183.7805056572,"seconds":27.393153886},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3296778047,"Relative Bias":-0.0467924813,"|Relative Bias|":0.0467924813,"метрика конкурса":0.376470286,"среднее по дням":0.3753554861,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":1,"features":246,"volume":1937103104.0,"absolute_error":638619895.4794979095,"signed_error":-90641860.3006782532,"seconds":27.40504695},{"выборка":"train","даты":25,"строки":125000,"WAPE":0.3330562777,"Relative Bias":-0.0001397281,"|Relative Bias|":0.0001397281,"метрика конкурса":0.3331960058,"среднее по дням":0.3526610599,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":1,"features":259,"volume":27932401664.0,"absolute_error":9303061841.1409072876,"signed_error":-3902941.8641471863,"seconds":27.288345443},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3285664889,"Relative Bias":-0.0509830964,"|Relative Bias|":0.0509830964,"метрика конкурса":0.3795495853,"среднее по дням":0.3783559337,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":1,"features":259,"volume":1937103104.0,"absolute_error":636467162.2561922073,"signed_error":-98759513.7331666946,"seconds":27.300788326},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3285988352,"Relative Bias":-0.000440987,"|Relative Bias|":0.000440987,"метрика конкурса":0.3290398222,"среднее по дням":0.3468353301,"variant":"direct","experiment":"baseline","horizon":8,"fold":2,"features":402,"volume":36539973632.0,"absolute_error":12006993200.9170913696,"signed_error":-16113653.051680088,"seconds":66.763180107},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3255836434,"Relative Bias":-0.0455111261,"|Relative Bias|":0.0455111261,"метрика конкурса":0.3710947695,"среднее по дням":0.3715818849,"variant":"direct","experiment":"baseline","horizon":8,"fold":2,"features":402,"volume":2093212544.0,"absolute_error":681515760.031627655,"signed_error":-95264459.1303577423,"seconds":66.780734431},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3298464692,"Relative Bias":-0.0004289429,"|Relative Bias|":0.0004289429,"метрика конкурса":0.3302754121,"среднее по дням":0.3483126307,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":2,"features":246,"volume":36539973632.0,"absolute_error":12052581716.146894455,"signed_error":-15673562.3611268997,"seconds":34.578231151},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3253759149,"Relative Bias":-0.0422209824,"|Relative Bias|":0.0422209824,"метрика конкурса":0.3675968973,"среднее по дням":0.369918974,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":2,"features":246,"volume":2093212544.0,"absolute_error":681080940.0129594803,"signed_error":-88377489.1080369949,"seconds":34.589966179},{"выборка":"train","даты":32,"строки":160000,"WAPE":0.3315853507,"Relative Bias":-0.0002332719,"|Relative Bias|":0.0002332719,"метрика конкурса":0.3318186226,"среднее по дням":0.3511167881,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":2,"features":259,"volume":36539973632.0,"absolute_error":12116120402.6476669312,"signed_error":-8523749.8996911049,"seconds":35.346890726},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3242562094,"Relative Bias":-0.0458413071,"|Relative Bias|":0.0458413071,"метрика конкурса":0.3700975165,"среднее по дням":0.368957057,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":2,"features":259,"volume":2093212544.0,"absolute_error":678737158.57837677,"signed_error":-95955598.095249176,"seconds":35.359657746},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3254835736,"Relative Bias":-0.0003225441,"|Relative Bias|":0.0003225441,"метрика конкурса":0.3258061176,"среднее по дням":0.3446110039,"variant":"direct","experiment":"baseline","horizon":8,"fold":3,"features":402,"volume":46040948736.0,"absolute_error":14985572793.2413024902,"signed_error":-14850235.1298904419,"seconds":81.719954671},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3227807049,"Relative Bias":-0.0271580822,"|Relative Bias|":0.0271580822,"метрика конкурса":0.3499387871,"среднее по дням":0.364854127,"variant":"direct","experiment":"baseline","horizon":8,"fold":3,"features":402,"volume":2133741056.0,"absolute_error":688730423.1847772598,"signed_error":-57948313.3598775864,"seconds":81.736303952},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3267377346,"Relative Bias":-0.0002810773,"|Relative Bias|":0.0002810773,"метрика конкурса":0.3270188119,"среднее по дням":0.3461641028,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":3,"features":246,"volume":46040948736.0,"absolute_error":15043315559.2287445068,"signed_error":-12941064.7690272331,"seconds":42.216439531},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322798267,"Relative Bias":-0.0225005215,"|Relative Bias|":0.0225005215,"метрика конкурса":0.3452987885,"среднее по дням":0.3640444325,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":3,"features":246,"volume":2133741056.0,"absolute_error":688767896.0660362244,"signed_error":-48010285.2061004639,"seconds":42.22813133},{"выборка":"train","даты":39,"строки":195000,"WAPE":0.3282360625,"Relative Bias":-0.0001369824,"|Relative Bias|":0.0001369824,"метрика конкурса":0.3283730449,"среднее по дням":0.3490083782,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":3,"features":259,"volume":46040948736.0,"absolute_error":15112299996.4470977783,"signed_error":-6306800.7200932503,"seconds":42.412977168},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3229935114,"Relative Bias":-0.0255747691,"|Relative Bias|":0.0255747691,"метрика конкурса":0.3485682805,"среднее по дням":0.3670424732,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":3,"features":259,"volume":2133741056.0,"absolute_error":689184496.9995527267,"signed_error":-54569933.2818002701,"seconds":42.425723459},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3228654476,"Relative Bias":-0.0003970123,"|Relative Bias|":0.0003970123,"метрика конкурса":0.3232624599,"среднее по дням":0.3427338585,"variant":"direct","experiment":"baseline","horizon":8,"fold":4,"features":402,"volume":56401149952.0,"absolute_error":18209981062.4067878723,"signed_error":-22391947.6921100616,"seconds":96.509054741},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3370196902,"Relative Bias":-0.0048453156,"|Relative Bias|":0.0048453156,"метрика конкурса":0.3418650058,"среднее по дням":0.3553246027,"variant":"direct","experiment":"baseline","horizon":8,"fold":4,"features":402,"volume":2057609216.0,"absolute_error":693454877.1491775513,"signed_error":-9969766.7834129333,"seconds":96.525822662},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.3238700933,"Relative Bias":-0.0003422084,"|Relative Bias|":0.0003422084,"метрика конкурса":0.3242123017,"среднее по дням":0.3438615236,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":4,"features":246,"volume":56401149952.0,"absolute_error":18266644232.5443725586,"signed_error":-19300945.0639028549,"seconds":50.610556915},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.337376396,"Relative Bias":-0.0001163815,"|Relative Bias|":0.0001163815,"метрика конкурса":0.3374927774,"среднее по дням":0.3545817312,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":4,"features":246,"volume":2057609216.0,"absolute_error":694188838.2943468094,"signed_error":-239467.6116075516,"seconds":50.622276769},{"выборка":"train","даты":46,"строки":230000,"WAPE":0.325160995,"Relative Bias":-0.0001940472,"|Relative Bias|":0.0001940472,"метрика конкурса":0.3253550422,"среднее по дням":0.3466743353,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":4,"features":259,"volume":56401149952.0,"absolute_error":18339452566.4551963806,"signed_error":-10944485.0171346664,"seconds":50.394564719},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3377736288,"Relative Bias":0.0020267527,"|Relative Bias|":0.0020267527,"метрика конкурса":0.3398003814,"среднее по дням":0.3522629497,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":4,"features":259,"volume":2057609216.0,"absolute_error":695006188.227347374,"signed_error":4170265.2844820023,"seconds":50.407201478},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3216305385,"Relative Bias":-0.0004030009,"|Relative Bias|":0.0004030009,"метрика конкурса":0.3220335393,"среднее по дням":0.3421885382,"variant":"direct","experiment":"baseline","horizon":8,"fold":5,"features":402,"volume":66924933120.0,"absolute_error":21525101671.9925765991,"signed_error":-26970806.1329545975,"seconds":112.854564156},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3357876381,"Relative Bias":0.0067074109,"|Relative Bias|":0.0067074109,"метрика конкурса":0.342495049,"среднее по дням":0.3628963849,"variant":"direct","experiment":"baseline","horizon":8,"fold":5,"features":402,"volume":2131451648.0,"absolute_error":715715148.8876390457,"signed_error":14296522.7438211441,"seconds":112.871382543},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3225175681,"Relative Bias":-0.0002912699,"|Relative Bias|":0.0002912699,"метрика конкурса":0.3228088381,"среднее по дням":0.3432711034,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":5,"features":246,"volume":66924933120.0,"absolute_error":21584466073.1664047241,"signed_error":-19493220.2063026428,"seconds":59.346546085},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3358943542,"Relative Bias":0.008128193,"|Relative Bias|":0.008128193,"метрика конкурса":0.3440225472,"среднее по дням":0.3628127468,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":5,"features":246,"volume":2131451648.0,"absolute_error":715942609.0423250198,"signed_error":17324851.2221574783,"seconds":59.358453617},{"выборка":"train","даты":53,"строки":265000,"WAPE":0.3237356386,"Relative Bias":-0.0001954306,"|Relative Bias|":0.0001954306,"метрика конкурса":0.3239310692,"среднее по дням":0.3459114582,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":5,"features":259,"volume":66924933120.0,"absolute_error":21665985355.8770446777,"signed_error":-13079180.125099659,"seconds":58.910603053},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3349385836,"Relative Bias":0.0089757853,"|Relative Bias|":0.0089757853,"метрика конкурса":0.343914369,"среднее по дням":0.3620995308,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":5,"features":259,"volume":2131451648.0,"absolute_error":713905430.2564725876,"signed_error":19131453.3008337021,"seconds":58.923248425},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3221720664,"Relative Bias":-0.0004656499,"|Relative Bias|":0.0004656499,"метрика конкурса":0.3226377163,"среднее по дням":0.3425991043,"variant":"direct","experiment":"baseline","horizon":8,"fold":6,"features":402,"volume":77243875328.0,"absolute_error":24885817429.5666351318,"signed_error":-35968600.3931512833,"seconds":127.303930331},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3447415535,"Relative Bias":0.0251950803,"|Relative Bias|":0.0251950803,"метрика конкурса":0.3699366338,"среднее по дням":0.3807251404,"variant":"direct","experiment":"baseline","horizon":8,"fold":6,"features":402,"volume":2059016192.0,"absolute_error":709828461.6868000031,"signed_error":51877079.7853679657,"seconds":127.320435299},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3231864654,"Relative Bias":-0.0003592077,"|Relative Bias|":0.0003592077,"метрика конкурса":0.3235456731,"среднее по дням":0.3439913891,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":6,"features":246,"volume":77243875328.0,"absolute_error":24964173538.9199447632,"signed_error":-27746591.8072214127,"seconds":67.166144681},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3449690273,"Relative Bias":0.0269152834,"|Relative Bias|":0.0269152834,"метрика конкурса":0.3718843107,"среднее по дням":0.3822511473,"variant":"direct","experiment":"thin_offsets","horizon":8,"fold":6,"features":246,"volume":2059016192.0,"absolute_error":710296834.0686979294,"signed_error":55419005.886218071,"seconds":67.177899659},{"выборка":"train","даты":60,"строки":300000,"WAPE":0.3243099741,"Relative Bias":-0.0002821304,"|Relative Bias|":0.0002821304,"метрика конкурса":0.3245921045,"среднее по дням":0.3460665887,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":6,"features":259,"volume":77243875328.0,"absolute_error":25050957700.0697288513,"signed_error":-21792840.9018268585,"seconds":68.025578299},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.3430266765,"Relative Bias":0.022648136,"|Relative Bias|":0.022648136,"метрика конкурса":0.3656748125,"среднее по дням":0.3785024587,"variant":"direct","experiment":"compact_with_additions","horizon":8,"fold":6,"features":259,"volume":2059016192.0,"absolute_error":706297502.0950784683,"signed_error":46632880.1785917282,"seconds":68.038775699}]'))
initial = previous.loc[previous['выборка'].eq('tuning')].groupby(['horizon','experiment'])['метрика конкурса'].mean().reset_index()
display(initial.sort_values(['horizon','метрика конкурса']).drop_duplicates('horizon'))


In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
end = times[-1]
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
selection_end = pd.Timestamp('2025-10-11')
if len(dates) < 14 or dates[-14].normalize() < selection_end or times[-1] < selection_end:
    raise ValueError('Недостаточно истории для отдельного test после подбора')
holdout_start = dates[-14]
selection_dates = dates[dates + 8 * STEP < selection_end]
if len(selection_dates) <= 49:
    raise ValueError('До 20 сентября недостаточно истории для train и трёх недель подбора')
folds = []
for start in range(len(selection_dates) - 42, len(selection_dates), 7):
    folds.append(dict(train=selection_dates[:start-7], val=selection_dates[start-7:start],
                      tuning=selection_dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    if lag_table is None:
        return correlation_block(frame,h,None,residual_signal,'residual_')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

In [ ]:
def stable_ratio(numerator, denominator):
    return numerator / np.maximum(np.abs(denominator), 1.0)


def available_statistics(window):
    valid = np.isfinite(window)
    count = valid.sum(axis=0)
    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count,
                     out=np.full(window.shape[1:], np.nan), where=count > 0)
    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count,
                         out=np.full_like(mean, np.nan), where=count > 0)
    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)
    return mean, np.sqrt(variance), spread, count


def add_experimental_features(frame, h, origins):
    additions = {}
    additions['x_time_cutoff_minutes'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))
    additions['x_profile_profile28'] = frame.profile28.to_numpy()
    additions['x_level_level24'] = level24[origins].ravel()
    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])
    current = target[origins]
    additions['x_anomaly_target_difference'] = (current - typical).ravel()
    additions['x_anomaly_target_ratio'] = stable_ratio(current, typical).ravel()
    for name in (TARGET, *STATUSES[:3]):
        short, long = means[name, 4][origins], means[name, 12][origins]
        additions[f'x_trend_{name}_difference'] = (short - long).ravel()
        additions[f'x_trend_{name}_ratio'] = stable_ratio(short, long).ravel()
        for steps in (4, 12):
            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])
            mean, std, spread, count = available_statistics(window)
            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]
            additions[f'x_volatility_{name}_std_{steps}'] = std.ravel()
            additions[f'x_volatility_{name}_range_{steps}'] = spread.ravel()
            additions[f'x_volatility_{name}_change_{steps}'] = change.ravel()
    for label, days in [('daily7', range(1,8)), ('weekly4', (7,14,21,28))]:
        window = np.stack([past(target, origins + h - day * 48) for day in days])
        mean, std, spread, count = available_statistics(window)
        additions[f'x_dispersion_{label}_std'] = std.ravel()
        additions[f'x_dispersion_{label}_relative_std'] = stable_ratio(std, mean).ravel()
        additions[f'x_dispersion_{label}_count'] = count.ravel()
    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)


def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    frame = add_experimental_features(frame, h, origins)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    minutes = times.hour * 60 + times.minute
    origins = np.flatnonzero(np.isin(minutes, TRAIN_MINUTES) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    if path.exists():
        print(f'h={h}: общие признаки загружены из кэша')
        return
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 35):
            batch = origins[start:start+35]
            frame = horizon_dataset(h, batch)
            frame['y'] = target[batch+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary,table.schema,compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    print(f'h={h}: история {len(origins)*len(routes):,}, модельных признаков {len(frame.columns) - 5}')
for h in range(1,9):
    write_horizon(h)

In [ ]:
def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()

In [ ]:
def model_target(frame):
    baseline = frame.profile28.to_numpy() if RESIDUAL else 0
    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)

def transformed_frame(frame, columns):
    result = frame[columns].drop(columns='route_id').astype(np.float32)
    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1
    bits = {f'route_bit_{i}': ((codes >> i) & 1).astype(np.float32)
            for i in range(max(1, int(len(routes)).bit_length()))}
    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()

def cached_lags(boundary):
    path = OUT / f'lags_{pd.Timestamp(boundary).strftime("%Y%m%dT%H%M%S")}.parquet'
    if path.exists():
        return pd.read_parquet(path)
    table = fit_lags(boundary)
    temporary = path.with_suffix('.partial')
    table.to_parquet(temporary, index=False)
    temporary.replace(path)
    return table

def fit_forest(frame, matrix, params):
    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type='numpy', **params)
    model.fit(matrix, model_target(frame))
    cp.cuda.get_current_stream().synchronize()
    return model

def predict_matrix(model, frame, matrix):
    result = np.empty(len(frame), dtype=np.float64)
    for start in range(0, len(frame), PREDICTION_BATCH):
        stop = min(start + PREDICTION_BATCH, len(frame))
        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)
        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0
        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)
    if not np.isfinite(result).all():
        raise ValueError('Некорректный прогноз')
    return result

EXPERIMENTS = {
    'baseline': dict(),
    'drop_constant': dict(drop_constant=True),
    'compact_availability': dict(compact_availability=True),
    'calendar_dayofweek': dict(calendar=True),
    'thin_offsets': dict(thin=True),
    'drop_network': dict(network='drop'),
    'thin_network': dict(network='thin'),
    'add_time': dict(add=['time']),
    'add_profile': dict(add=['profile']),
    'add_level24': dict(add=['level']),
    'add_anomaly': dict(add=['anomaly']),
    'add_trends': dict(add=['trend']),
    'add_volatility': dict(add=['volatility']),
    'add_seasonal_dispersion': dict(add=['dispersion']),
    'all_additions': dict(add=['time','profile','level','anomaly','trend','volatility','dispersion']),
    'compact': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin'),
    'compact_with_additions': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin',
                                   add=['time','profile','level','anomaly','trend','volatility','dispersion'])
}


def experiment_columns(names, spec):
    result = []
    for name in names:
        if name.startswith('x_'):
            if any(name.startswith(f'x_{group}_') for group in spec.get('add', [])):
                result.append(name)
            continue
        if spec.get('drop_constant') and name == 'selected_status6_pair_count':
            continue
        if spec.get('compact_availability') and 'signal_available' in name:
            continue
        if spec.get('calendar') and name in ('weekday_sin','weekday_cos','is_weekend'):
            continue
        if spec.get('network') == 'drop' and name.startswith('network_'):
            continue
        if spec.get('network') == 'thin' and name.startswith('network_') and '_offset_' in name:
            continue
        if spec.get('thin') and not name.startswith('slot_') and re.search(r'_offset_m[134]$', name):
            continue
        result.append(name)
    if spec.get('compact_availability'):
        for name in ('status6_lag_valid', 'residual_status6_lag_valid'):
            if name in names and name not in result:
                result.append(name)
    return result

## Все сочетания подготовки и признаков на h2 и h6
Отсечение удаляет начало train до доступности всех смещений первого недельного лага. Остальные NaN сохраняются до заполнения. Маршрутная медиана рассчитывается только на train, при отсутствии наблюдений используется общая медиана train. История для вычисления лагов сохраняется полной. Метрики train и следующего временного окна выводятся после каждого обучения.

In [ ]:

def matrices(h, fold, trimmed, route_median, experiment, final=False):
    boundary = holdout_start.normalize() if final else fold['val'][0].normalize()
    eval_dates = dates[-14:] if final else fold['tuning']
    history = pd.read_parquet(OUT / f'history_h{h}.parquet')
    history = history.loc[np.isfinite(history.profile28)].copy()
    train = history.loc[history.timestamp < boundary].copy()
    valid = history.loc[history.cutoff.isin(eval_dates)].copy()
    del history
    if trimmed:
        names = [offset_name('slot_days_back_7', k) for k in WINDOW_OFFSETS]
        ready = train[names].notna().all(axis=1)
        first = train.loc[ready].groupby('route_id').cutoff.min()
        starts = train.route_id.map(first)
        train = train.loc[starts.notna() & train.cutoff.ge(starts)].copy()
    if train.empty or valid.empty or train.timestamp.max() >= valid.cutoff.min().normalize():
        raise ValueError('Некорректное временное разделение.')
    if len(valid) != len(eval_dates)*len(routes):
        raise ValueError('Потеряны строки оценки.')
    lag_table = cached_lags(boundary)
    frames, encoded = [], []
    for frame in (train, valid):
        frame = add_correlation_features(frame, h, lag_table)
        x = transformed_frame(frame, model_columns(frame))
        columns = experiment_columns(list(x.columns), EXPERIMENTS[experiment])
        encoded.append(x[columns].replace([np.inf,-np.inf],np.nan))
        frames.append(frame[['cutoff','timestamp','route_id','y','profile28']].copy())
    del train, valid, frame, x
    if route_median:
        medians = encoded[0].groupby(frames[0].route_id).median()
        for i in range(2):
            fill = medians.reindex(frames[i].route_id.to_numpy())
            fill.index = encoded[i].index
            encoded[i] = encoded[i].fillna(fill)
    prep = SimpleImputer(strategy='median', keep_empty_features=True)
    arrays = [prep.fit_transform(encoded[0]).astype(np.float32), prep.transform(encoded[1]).astype(np.float32)]
    return frames, arrays

def fit_once(h, fold_number, trimmed, route_median, experiment, params, label, final=False):
    folder = RESULTS / label / f'h{h}_fold{fold_number}'
    folder.mkdir(parents=True, exist_ok=True)
    marker = folder / 'metrics.json'
    signature = dict(trimmed=trimmed, route_median=route_median, experiment=experiment, params=params, final=final)
    if marker.exists():
        saved = json.loads(marker.read_text())
        if saved['signature'] != signature:
            raise ValueError(f'Конфликт кэша {folder}')
        return saved['rows'], pd.read_parquet(folder/'predictions.parquet')
    frames, arrays = matrices(h, None if final else folds[fold_number-1], trimmed, route_median, experiment, final)
    started = time.perf_counter()
    model = fit_forest(frames[0], arrays[0], params)
    rows = []
    for frame, matrix, split in zip(frames, arrays, ('train','test' if final else 'tuning')):
        pred = predict_matrix(model, frame, matrix)
        rows.append(dict(horizon=h, fold=fold_number, trimmed=trimmed, route_median=route_median,
                         experiment=experiment, **score_row(frame,pred,split)))
        if split != 'train':
            predictions = frame.assign(prediction=pred, horizon=h, fold=fold_number)
    save_frame(folder/'predictions.parquet', predictions)
    save_json(marker, dict(signature=signature, rows=rows, seconds=time.perf_counter()-started))
    display(pd.DataFrame(rows))
    del model, frames, arrays
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    return rows, predictions

all_rows = []
for h in HORIZONS:
    for trimmed in (False, True):
        for route_median in (False, True):
            for experiment in [initial.sort_values(['horizon','метрика конкурса']).drop_duplicates('horizon').set_index('horizon').loc[h,'experiment']]:
                label = f'comparison/t{int(trimmed)}_r{int(route_median)}_{experiment}'
                for number in range(1,7):
                    if not trimmed and not route_median:
                        cached = previous.loc[previous.horizon.eq(h) & previous.fold.eq(number)
                                              & previous.experiment.eq(experiment)]
                        rows = cached.assign(trimmed=False, route_median=False).to_dict('records')
                    else:
                        rows, _ = fit_once(h, number, trimmed, route_median, experiment,
                                           BASE_PARAMS_BY_HORIZON[h], label)
                    all_rows.extend(rows)
                    pd.DataFrame(all_rows).to_csv(RESULTS/'comparison.csv',index=False)
scores = pd.DataFrame(all_rows)
ranking = (scores.loc[scores['выборка'].eq('tuning')]
           .groupby(['horizon','trimmed','route_median','experiment'])['метрика конкурса']
           .agg(['mean','std','max']).reset_index()
           .sort_values(['horizon','mean','trimmed','route_median','experiment']))
selected = ranking.drop_duplicates('horizon').set_index('horizon')
ranking.to_csv(RESULTS/'preprocessing_ranking.csv',index=False)
display(selected)

initial_winners = initial.sort_values(['horizon','метрика конкурса']).drop_duplicates('horizon').set_index('horizon')
for h in range(1,9):
    if h not in selected.index:
        selected.loc[h,['trimmed','route_median','experiment']] = [False,False,initial_winners.loc[h,'experiment']]
if not (selected.experiment == initial_winners.experiment.reindex(selected.index)).all():
    raise ValueError('Изменился фиксированный набор признаков.')
selected.to_csv(RESULTS/'preprocessing_selected_before_search.csv')


## Optuna отдельно для каждого горизонта

In [ ]:

def suggest(trial):
    return dict(n_estimators=trial.suggest_categorical('n_estimators',[150,300]),
                max_depth=trial.suggest_categorical('max_depth',[6,8,10,12,16]),
                min_samples_leaf=trial.suggest_int('min_samples_leaf',8,64),
                max_features=trial.suggest_float('max_features',0.25,1.),
                max_samples=trial.suggest_float('max_samples',0.5,1.))


studies = {}
for h in range(1,9):
    study = optuna.create_study(study_name=f'h{h}', direction='minimize',
        storage=f'sqlite:///{(RESULTS/"optuna.sqlite3").resolve()}',
        sampler=optuna.samplers.TPESampler(seed=SEED), load_if_exists=True)
    if not study.trials:
        study.enqueue_trial(BASE_PARAMS_BY_HORIZON[h])
    for trial in study.get_trials(states=(optuna.trial.TrialState.RUNNING,)):
        study.tell(trial.number,state=optuna.trial.TrialState.FAIL)
        if trial.params:
            study.enqueue_trial(trial.params)
    studies[h] = study

for round_number in range(N_TRIALS):
    for h in range(1,9):
        study = studies[h]
        completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
        if len(completed) > round_number:
            continue
        if completed and time.monotonic() >= SEARCH_DEADLINE:
            continue
        spec = selected.loc[h]
        trimmed, route_median, experiment = bool(spec.trimmed), bool(spec.route_median), spec.experiment
        def objective(trial):
            params = suggest(trial)
            scores = []
            for number in range(1,7):
                rows,_ = fit_once(h,number,trimmed,route_median,experiment,params,
                                  f'search/trial{trial.number}')
                scores.append(rows[-1]['метрика конкурса'])
            return float(np.mean(scores))
        study.optimize(objective,n_trials=1,n_jobs=1,gc_after_trial=True)
    if time.monotonic() >= SEARCH_DEADLINE:
        break

best_params, raw_parts = {}, []
for h in range(1,9):
    study = studies[h]
    best_params[h] = study.best_params
    spec = selected.loc[h]
    for number in range(1,7):
        _,part = fit_once(h,number,bool(spec.trimmed),bool(spec.route_median),spec.experiment,
                          study.best_params,f'search/trial{study.best_trial.number}')
        raw_parts.append(part)
save_json(RESULTS/'best_params.json',best_params)
oof = pd.concat(raw_parts,ignore_index=True)
save_frame(RESULTS/'tuning_predictions.parquet',oof)


## Калибровки · 19 вариантов
Исходный прогноз и шесть схем с силой 50%, 75%, 100%. Первые два tuning-окна дают начальную историю поправок. Каждое следующее окно корректируется по предыдущим. Выбираем по средней недельной метрике объединённых восьми горизонтов. Группы маршрутов определяются по среднему факту только на истории калибровки. Для групповых множителей используем сглаживание к коэффициенту горизонта. Изотоническая поправка обучается отдельно по горизонту.

In [ ]:

def metric(y,p):
    return float((np.abs(p-y).sum()+abs((p-y).sum()))/max(y.sum(),1e-12))

def fit_calibration(history, kind):
    state = dict(kind=kind)
    if kind == 'none':
        return state
    state['models'] = {}
    groups = [('all',history)] if kind == 'global' else list(history.groupby('horizon'))
    for key, frame in groups:
        p,y = frame.prediction.to_numpy(),frame.y.to_numpy()
        scale = max(float(y.mean()),1.)
        factor = float(y.sum()/p.sum()) if p.sum()>0 else 1.
        entry = dict(factor=factor)
        if kind in ('shift','affine'):
            initial = [0.] if kind=='shift' else [1.,0.]
            def loss(theta):
                a,b = (1.,theta[0]) if kind=='shift' else theta
                return metric(y,np.maximum(0,a*p+b*scale))
            result = minimize(loss,initial,method='Powell',
                bounds=[(-2.,2.)] if kind=='shift' else [(0.,3.),(-2.,2.)])
            theta = result.x if np.isfinite(result.fun) and result.fun<loss(initial) else initial
            a,b = (1.,theta[0]) if kind=='shift' else theta
            entry.update(a=float(a),b=float(b*scale))
        if kind == 'group':
            volume = frame.groupby('route_id').y.mean()
            edges = np.unique(np.quantile(volume,[1/3,2/3]))
            mapping = {route:int(np.searchsorted(edges,value)) for route,value in volume.items()}
            entry['mapping'] = mapping
            entry['factors'] = {}
            grouped = frame.assign(group=frame.route_id.map(mapping))
            for group, part in grouped.groupby('group'):
                ratio = float(part.y.sum()/part.prediction.sum()) if part.prediction.sum()>0 else factor
                weight = len(part)/(len(part)+200.)
                entry['factors'][group] = weight*ratio+(1-weight)*factor
        if kind == 'nonlinear':
            entry['model'] = IsotonicRegression(y_min=0,out_of_bounds='clip').fit(p,y)
        state['models'][key] = entry
    return state

def apply_calibration(state, frame, strength):
    p = frame.prediction.to_numpy(dtype=float)
    if state['kind'] == 'none':
        return p
    result = p.copy()
    groups = [('all',np.arange(len(frame)))] if state['kind']=='global' else frame.groupby('horizon').indices.items()
    for key, index in groups:
        entry = state['models'][key]
        raw = p[index]
        kind = state['kind']
        if kind in ('global','horizon'):
            corrected = raw*entry['factor']
        elif kind in ('shift','affine'):
            corrected = raw*entry['a']+entry['b']
        elif kind == 'group':
            keys = frame.iloc[index].route_id.map(entry['mapping'])
            factors = keys.map(entry['factors']).fillna(entry['factor']).to_numpy()
            corrected = raw*factors
        else:
            corrected = entry['model'].predict(raw)
        result[index] = np.maximum(0,raw+strength*(np.maximum(0,corrected)-raw))
    return result

candidates = [('none',0.)]+[(k,v) for k in ('global','horizon','shift','affine','group','nonlinear') for v in (.5,.75,1.)]
calibration_rows = []
for kind,strength in candidates:
    for number in range(3,7):
        valid = oof.loc[oof.fold.eq(number)].copy()
        past = oof.loc[oof.fold.lt(number) & oof.timestamp.lt(valid.cutoff.min().normalize())].copy()
        state = fit_calibration(past,kind)
        pred = apply_calibration(state,valid,strength)
        calibration_rows.append(dict(kind=kind,strength=strength,fold=number,**score_row(valid,pred,'tuning')))
calibration_details = pd.DataFrame(calibration_rows)
calibration_details.to_csv(RESULTS/'calibration_windows.csv',index=False)
calibration_ranking = (calibration_details.groupby(['kind','strength'],sort=False)['метрика конкурса']
                       .mean().reset_index().sort_values('метрика конкурса',kind='stable'))
display(calibration_ranking)
calibration_ranking.to_csv(RESULTS/'calibration_ranking.csv',index=False)
winner = calibration_ranking.iloc[0]
frozen = dict(kind=winner.kind,strength=float(winner.strength))
save_json(RESULTS/'calibration_frozen_before_test.json',frozen)
calibration = fit_calibration(oof,frozen['kind'])
joblib.dump(calibration,RESULTS/'calibration.joblib')


## Независимый локальный test
Признаки, обработка train, параметры и калибровка зафиксированы до этой ячейки. Обучаем выбранные леса на всей истории до начала test. Независимая оценка test получена до переобучения на всей размеченной истории. После сохранения независимой оценки выполняется переобучение всех выбранных лесов на всей размеченной истории и сборка submission.csv.

In [ ]:

test_parts, independent = [], []
for h in range(1,9):
    spec = selected.loc[h]
    rows, part = fit_once(h,0,bool(spec.trimmed),bool(spec.route_median),spec.experiment,
                          best_params[h],'independent',final=True)
    test_parts.append(part)
    independent.extend(rows)
pd.DataFrame(independent).to_csv(RESULTS/'independent_train_test.csv',index=False)
test = pd.concat(test_parts,ignore_index=True)
test['calibrated'] = apply_calibration(calibration,test,frozen['strength'])
save_frame(RESULTS/'test_predictions.parquet',test)
summary = []
for h,frame in [('all',test),*list(test.groupby('horizon'))]:
    for label,column in [('raw','prediction'),('selected_calibration','calibrated')]:
        summary.append(dict(horizon=h,model=label,**score_row(frame,frame[column].to_numpy(),'test')))
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(RESULTS/'independent_calibration_metrics.csv',index=False)
print('Независимая оценка test получена до переобучения на всей размеченной истории.')
print(RESULTS.resolve())


## Финальное переобучение и submission.csv
Подключите test_solo_track.parquet. Все восемь лесов переобучаются на всей размеченной истории своего горизонта с выбранной обработкой строк и признаков. Калибровка фиксируется до локального test и применяется после переобучения. Локальный test входит в финальную размеченную историю. Независимые метрики выше получены до этого переобучения.

In [ ]:

TEST_PATH = TRAIN_PATH.with_name('test_solo_track.parquet')
if not TEST_PATH.is_file():
    matches = list(Path('/kaggle/input').rglob('test_solo_track.parquet'))
    if len(matches)!=1:
        raise FileNotFoundError('Укажите TEST_PATH.')
    TEST_PATH = matches[0]
query = pd.read_parquet(TEST_PATH)
if query.id.duplicated().any() or query.duplicated(['route_id','timestamp']).any():
    raise ValueError('Повторяющиеся ключи test.')
observed_end = times[-1]
forecast_origin = observed_end
query['horizon'] = ((query.timestamp-forecast_origin)/STEP).astype(int)
if not query.horizon.isin(range(1,9)).all() or not (query.timestamp == forecast_origin + query.horizon*STEP).all():
    raise ValueError('Test должен содержать восемь получасовых горизонтов после конца истории.')
original_times = times
times = pd.date_range(times[0], query.timestamp.max(), freq=STEP)
for name in values:
    values[name] = np.pad(values[name],((0,len(times)-len(original_times)),(0,0)),constant_values=np.nan)


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    if lag_table is None:
        return correlation_block(frame,h,None,residual_signal,'residual_')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

In [ ]:
def stable_ratio(numerator, denominator):
    return numerator / np.maximum(np.abs(denominator), 1.0)

def available_statistics(window):
    valid = np.isfinite(window)
    count = valid.sum(axis=0)
    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count,
                     out=np.full(window.shape[1:], np.nan), where=count > 0)
    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count,
                         out=np.full_like(mean, np.nan), where=count > 0)
    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)
    return mean, np.sqrt(variance), spread, count

def add_experimental_features(frame, h, origins):
    additions = {}
    additions['x_time_cutoff_minutes'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))
    additions['x_profile_profile28'] = frame.profile28.to_numpy()
    additions['x_level_level24'] = level24[origins].ravel()
    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])
    current = target[origins]
    additions['x_anomaly_target_difference'] = (current - typical).ravel()
    additions['x_anomaly_target_ratio'] = stable_ratio(current, typical).ravel()
    for name in (TARGET, *STATUSES[:3]):
        short, long = means[name, 4][origins], means[name, 12][origins]
        additions[f'x_trend_{name}_difference'] = (short - long).ravel()
        additions[f'x_trend_{name}_ratio'] = stable_ratio(short, long).ravel()
        for steps in (4, 12):
            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])
            mean, std, spread, count = available_statistics(window)
            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]
            additions[f'x_volatility_{name}_std_{steps}'] = std.ravel()
            additions[f'x_volatility_{name}_range_{steps}'] = spread.ravel()
            additions[f'x_volatility_{name}_change_{steps}'] = change.ravel()
    for label, days in [('daily7', range(1,8)), ('weekly4', (7,14,21,28))]:
        window = np.stack([past(target, origins + h - day * 48) for day in days])
        mean, std, spread, count = available_statistics(window)
        additions[f'x_dispersion_{label}_std'] = std.ravel()
        additions[f'x_dispersion_{label}_relative_std'] = stable_ratio(std, mean).ravel()
        additions[f'x_dispersion_{label}_count'] = count.ravel()
    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)

def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    frame = add_experimental_features(frame, h, origins)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    minutes = times.hour * 60 + times.minute
    origins = np.flatnonzero(np.isin(minutes, TRAIN_MINUTES) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    if path.exists():
        print(f'h={h}: общие признаки загружены из кэша')
        return
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 35):
            batch = origins[start:start+35]
            frame = horizon_dataset(h, batch)
            frame['y'] = target[batch+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary,table.schema,compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    print(f'h={h}: история {len(origins)*len(routes):,}, модельных признаков {len(frame.columns) - 5}')

In [ ]:

submission_parts = []
full_boundary = observed_end + STEP
lag_table = fit_lags(full_boundary)
origin_index = times.get_loc(forecast_origin)
for h in range(1,9):
    spec = selected.loc[h]
    folder = RESULTS/'full_refit'/f'h{h}'
    folder.mkdir(parents=True,exist_ok=True)
    marker = folder/'complete.json'
    if marker.exists():
        submission_parts.append(pd.read_parquet(folder/'prediction.parquet'))
        continue
    train = pd.read_parquet(OUT/f'history_h{h}.parquet')
    train = train.loc[np.isfinite(train.profile28) & np.isfinite(train.y)].copy()
    if bool(spec.trimmed):
        names = [offset_name('slot_days_back_7', k) for k in WINDOW_OFFSETS]
        ready = train[names].notna().all(axis=1)
        first = train.loc[ready].groupby('route_id').cutoff.min()
        starts = train.route_id.map(first)
        train = train.loc[starts.notna() & train.cutoff.ge(starts)].copy()
    future = horizon_dataset(h,np.array([origin_index]))
    future['y'] = np.nan
    frames, encoded = [], []
    for frame in (train,future):
        frame = add_correlation_features(frame,h,lag_table)
        x = transformed_frame(frame,model_columns(frame))
        names = experiment_columns(list(x.columns),EXPERIMENTS[spec.experiment])
        encoded.append(x[names].replace([np.inf,-np.inf],np.nan))
        frames.append(frame[['cutoff','timestamp','route_id','y','profile28']].copy())
    del train,future,frame,x
    route_medians = None
    if bool(spec.route_median):
        route_medians = encoded[0].groupby(frames[0].route_id).median()
        for i in range(2):
            fill = route_medians.reindex(frames[i].route_id.to_numpy())
            fill.index = encoded[i].index
            encoded[i] = encoded[i].fillna(fill)
    prep = SimpleImputer(strategy='median',keep_empty_features=True)
    train_x = prep.fit_transform(encoded[0]).astype(np.float32)
    future_x = prep.transform(encoded[1]).astype(np.float32)
    del encoded
    model = fit_forest(frames[0],train_x,best_params[h])
    train_pred = predict_matrix(model,frames[0],train_x)
    display(pd.DataFrame([dict(horizon=h,**score_row(frames[0],train_pred,'full_refit_train'))]))
    part = frames[1].assign(horizon=h,prediction=predict_matrix(model,frames[1],future_x))
    part['calibrated'] = apply_calibration(calibration,part,frozen['strength'])
    save_frame(folder/'prediction.parquet',part)
    joblib.dump(dict(model=model,imputer=prep,route_medians=route_medians,columns=names,
                     params=best_params[h],spec=spec.to_dict()),folder/'model.joblib')
    save_json(marker,dict(horizon=h,rows_train=len(frames[0]),labels_through=str(observed_end)))
    submission_parts.append(part)
    del model,frames,train_x,future_x,train_pred
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
predictions = pd.concat(submission_parts,ignore_index=True)
submission = query.merge(predictions[['route_id','timestamp','calibrated']],on=['route_id','timestamp'],
                          how='left',validate='one_to_one').sort_values('id')
if len(submission)!=len(query) or not np.isfinite(submission.calibrated).all() or (submission.calibrated<0).any():
    raise ValueError('Некорректный сабмит.')
submission = submission[['id','calibrated']].rename(columns={'calibrated':'y_pred'})
submission.to_csv(RESULTS/'submission.csv',index=False)
display(submission.head())
print(RESULTS/'submission.csv')
print('Независимая оценка test получена до финального переобучения на всей размеченной истории.')
